# Random Forest Regressor — Thermal Predictor

Modelo de regressão multi-output que prevê a temperatura de cada termopar
(`tc_name`) de uma `key_area` a partir das sete variáveis de configuração do ensaio.

**Entrada:** planilhas em formato wide geradas por `tratamento_final_dos_dados.ipynb`
(`data/<key_area>_data.csv`) — uma linha por run, uma coluna por termopar.

As sete planilhas têm o mesmo formato, mas diferem em número de linhas (55 a 73),
número de termopares (4 a 37) e, principalmente, em **quais sensores foram medidos
em quais runs**. Por isso o pipeline abaixo é escrito em funções que se adaptam ao
arquivo carregado, sem nenhum parâmetro fixo por área.

Rode as células em ordem. Para trocar de área, altere apenas `CAMINHO_PLANILHA`
na Célula 2 e execute o notebook novamente. A última célula roda as sete de uma vez.

## 1. Bibliotecas e configuração global

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
SEMENTE = 42

## 2. Parâmetros e leitura da planilha

Formato wide: cada termopar é uma coluna-alvo independente. Tudo que não está em
`VARIAVEIS_INDEPENDENTES` é candidato a alvo.

Duas checagens são feitas na leitura, porque nem toda coluna extra é um termopar
válido: colunas **sem nome** (`Unnamed: N`, geradas quando o `tc_name` original
estava em branco — é o caso de `ct_knobs_data.csv`) e colunas **inteiramente
vazias** são descartadas com aviso, em vez de entrarem como alvo do modelo.

In [ ]:
# ============================================================
# CÉLULA 2 — PARÂMETROS E LEITURA DA PLANILHA
# Formato wide: cada termopar é uma coluna-alvo independente
# ============================================================

# Troque aqui para rodar o modelo em outra key_area:
#   ../data/mainback_data.csv        ../data/ipc_comp_data.csv
#   ../data/coil_temps_data.csv      ../data/ipc_cooling_data.csv
#   ../data/main_vent_sys_data.csv   ../data/oven_hmi_data.csv
#   ../data/ct_knobs_data.csv
CAMINHO_PLANILHA = "../data/main_vent_sys_data.csv"

VARIAVEIS_INDEPENDENTES = [
    "blower_direction",
    "housing",
    "board_position",
    "run_case",
    "losses",
    "deflector_hole",
    "material_coil_support",
]

# Um termopar só entra como alvo se tiver leitura em pelo menos esta fração dos
# runs. Sensores abaixo disso são removidos como COLUNA — descartá-los custa um
# alvo, enquanto exigir que estejam presentes custaria quase todos os runs.
COBERTURA_MINIMA_ALVO = 0.50

# Número de folds do GridSearchCV. É reduzido automaticamente se o conjunto de
# treino for pequeno demais (ver Célula 5).
N_FOLDS_DESEJADO = 3


def identificar_alvos(df, independentes):
    """Separa as colunas-alvo válidas das colunas que não representam termopar."""
    alvos, descartadas = [], []
    for coluna in df.columns:
        if coluna in independentes:
            continue
        if str(coluna).startswith("Unnamed:") or not str(coluna).strip():
            descartadas.append((coluna, "coluna sem nome de termopar"))
        elif df[coluna].isna().all():
            descartadas.append((coluna, "coluna inteiramente vazia"))
        else:
            alvos.append(coluna)
    return alvos, descartadas


def resumir_alvos(df, alvos):
    """Tabela de cobertura e estatística descritiva de cada termopar."""
    return pd.DataFrame([
        {
            "termopar": alvo,
            "leituras": int(df[alvo].notna().sum()),
            "cobertura": df[alvo].notna().mean(),
            "min": df[alvo].min(),
            "max": df[alvo].max(),
            "média": df[alvo].mean(),
            "desvio": df[alvo].std(),
        }
        for alvo in alvos
    ]).sort_values("cobertura", ascending=False, ignore_index=True)


def listar_alvos_duplicados(df, alvos):
    """Pares de termopares com leitura idêntica (possível duplicação de sensor)."""
    return [
        (alvo_a, alvo_b)
        for i, alvo_a in enumerate(alvos)
        for alvo_b in alvos[i + 1:]
        if df[alvo_a].equals(df[alvo_b])
    ]


dados = pd.read_csv(CAMINHO_PLANILHA)
VARIAVEIS_ALVO, COLUNAS_DESCARTADAS = identificar_alvos(dados, VARIAVEIS_INDEPENDENTES)

print(f"Arquivo: {CAMINHO_PLANILHA}")
print(f"Linhas: {len(dados)} | Colunas: {dados.shape[1]}")

if COLUNAS_DESCARTADAS:
    print("\nColunas ignoradas (não são termopares válidos):")
    for coluna, motivo in COLUNAS_DESCARTADAS:
        print(f"   {coluna!r}: {motivo}")

print(f"\nTermopares candidatos ({len(VARIAVEIS_ALVO)}):")
print(resumir_alvos(dados, VARIAVEIS_ALVO).round(2).to_string(index=False))

print(f"\nValores nulos nos alvos: {int(dados[VARIAVEIS_ALVO].isna().sum().sum())}")

duplicadas = listar_alvos_duplicados(dados, VARIAVEIS_ALVO)
if duplicadas:
    print("\nATENÇÃO — colunas-alvo com valores idênticos:")
    for alvo_a, alvo_b in duplicadas:
        print(f"   {alvo_a}  ==  {alvo_b}")

dados.head()

## 3. Seleção de alvos, padronização das categorias e codificação

Esta é a célula que precisou mudar para o notebook funcionar nas sete áreas.

A versão anterior fazia `dropna(subset=VARIAVEIS_ALVO)` direto, ou seja, exigia que
**todos** os termopares tivessem leitura na mesma linha. Isso é *listwise deletion*:
basta um sensor pouco medido para eliminar quase todos os runs. Em
`main_vent_sys_data.csv`, dois sensores (`AC Blower Inlet Avg` e `Max`) só foram
medidos em 4 dos 56 runs, então a base caía de 56 para 4 linhas — 2 no treino, menos
que os 3 folds do `GridSearchCV`, o que produzia o `ValueError`.

A correção inverte a ordem: primeiro descarta o **sensor** com cobertura baixa,
depois descarta as **linhas** que ainda tenham buraco entre os alvos que sobraram.
No mesmo arquivo, a base passa de 4 para 56 linhas ao custo de 2 termopares.

In [ ]:
# ============================================================
# CÉLULA 3 — SELEÇÃO DE ALVOS, PADRONIZAÇÃO E CODIFICAÇÃO ORDINAL
# ============================================================
GRAFIAS_DIVERGENTES = {
    "blower_direction": {"To back": "To Back", "To front": "To Front"},
    "losses":           {"No losses": "No Losses"},
    "run_case":         {"Broil Case 2": "Broil C2", "Broil Qual": "Broil Qualitative", "Qualitative": "Broil Qualitative"},
    "housing":          {"Open - Updated": "Open"}
}


def filtrar_alvos_por_cobertura(df, alvos, cobertura_minima):
    """Mantém apenas os termopares medidos em uma fração mínima dos runs."""
    mantidos, removidos = [], []
    for alvo in alvos:
        cobertura = df[alvo].notna().mean()
        if cobertura >= cobertura_minima:
            mantidos.append(alvo)
        else:
            removidos.append((alvo, int(df[alvo].notna().sum()), cobertura))
    return mantidos, removidos


def padronizar_categorias(df, independentes, correcoes_por_variavel):
    """Unifica grafias divergentes e transforma nulos em categoria explícita."""
    df = df.copy()
    for variavel, correcoes in correcoes_por_variavel.items():
        if variavel in df.columns:
            df[variavel] = df[variavel].replace(correcoes)
    for variavel in independentes:
        df[variavel] = df[variavel].fillna("Nao Informado").astype(str)
    return df


def codificar_ordinal(df, independentes):
    """Codificação ordinal (0, 1, 2, ...) em ordem alfabética por variável."""
    X = df[independentes].copy()
    dicionario = {}
    for variavel in independentes:
        codigos = {categoria: i for i, categoria in enumerate(sorted(X[variavel].unique()))}
        dicionario[variavel] = codigos
        X[variavel] = X[variavel].map(codigos)
    return X, dicionario


# 1) Descarta o SENSOR pouco medido (coluna), não o run (linha).
VARIAVEIS_ALVO, alvos_removidos = filtrar_alvos_por_cobertura(
    dados, VARIAVEIS_ALVO, COBERTURA_MINIMA_ALVO
)

if alvos_removidos:
    print(f"Termopares removidos por cobertura < {COBERTURA_MINIMA_ALVO:.0%}:")
    for alvo, leituras, cobertura in alvos_removidos:
        print(f"   {alvo:40s} {leituras:3d}/{len(dados)} runs ({cobertura:.1%})")
    print()

if not VARIAVEIS_ALVO:
    raise ValueError(
        f"Nenhum termopar atingiu a cobertura mínima de {COBERTURA_MINIMA_ALVO:.0%}. "
        f"Reduza COBERTURA_MINIMA_ALVO na Célula 2 para modelar este arquivo."
    )

# 2) Só agora descarta as LINHAS com buraco entre os alvos que sobraram.
dados_tratados = dados.dropna(subset=VARIAVEIS_ALVO).copy()

linhas_removidas = len(dados) - len(dados_tratados)
if linhas_removidas:
    print(f"Linhas removidas por alvo ausente: {linhas_removidas}\n")

dados_tratados = padronizar_categorias(
    dados_tratados, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES
)
X, dicionario_codificacao = codificar_ordinal(dados_tratados, VARIAVEIS_INDEPENDENTES)
Y = dados_tratados[VARIAVEIS_ALVO]

print("Dicionário de codificação:\n")
for variavel, codigos in dicionario_codificacao.items():
    print(f"{variavel} ({len(codigos)} categorias)")
    for categoria, codigo in codigos.items():
        print(f"   {codigo:>2d} = {categoria}")
    print()

print(f"Matriz de features: {X.shape}")
print(f"Colunas-alvo: {len(VARIAVEIS_ALVO)}")
print(f"Linhas úteis: {len(dados_tratados)} de {len(dados)}")
print(f"Combinações distintas de configuração: {X.drop_duplicates().shape[0]}")

## 4. Separação treino (70%) / teste (20%) / verificação (10%)

`Y` é matricial: uma coluna por termopar, lida de `dados_tratados` para permanecer
alinhada com as linhas de `X`.

A célula checa antes da divisão se sobraram linhas suficientes para formar os três
conjuntos e ainda alimentar a validação cruzada, e falha com uma mensagem que diz o
que ajustar — em vez do `ValueError` genérico do `GridSearchCV` duas células adiante.

In [ ]:
# ============================================================
# CÉLULA 4 — SEPARAÇÃO TREINO (70%) / TESTE (20%) / VERIFICAÇÃO (10%)
# Y é matricial (uma coluna por termopar)
# ============================================================
PROPORCAO_TREINO = 0.70
PROPORCAO_TESTE = 0.20
PROPORCAO_VERIFICACAO = 0.10

MINIMO_DE_LINHAS = 10
if len(X) < MINIMO_DE_LINHAS:
    raise ValueError(
        f"Apenas {len(X)} linhas úteis após o tratamento — insuficiente para dividir em "
        f"treino/teste/verificação e rodar validação cruzada. Verifique a Célula 3: "
        f"provavelmente há termopares esparsos derrubando linhas. Considere reduzir "
        f"COBERTURA_MINIMA_ALVO (hoje {COBERTURA_MINIMA_ALVO:.0%}) na Célula 2."
    )

X_treino, X_restante, Y_treino, Y_restante = train_test_split(
    X, Y,
    train_size=PROPORCAO_TREINO,
    random_state=SEMENTE,
)

proporcao_verificacao_no_restante = (
    PROPORCAO_VERIFICACAO / (PROPORCAO_TESTE + PROPORCAO_VERIFICACAO)
)

X_teste, X_verificacao, Y_teste, Y_verificacao = train_test_split(
    X_restante, Y_restante,
    test_size=proporcao_verificacao_no_restante,
    random_state=SEMENTE,
)

print(f"Treino:       {len(X_treino):3d} linhas ({len(X_treino)/len(X):.1%})")
print(f"Teste:        {len(X_teste):3d} linhas ({len(X_teste)/len(X):.1%})")
print(f"Verificação:  {len(X_verificacao):3d} linhas ({len(X_verificacao)/len(X):.1%})")
print(f"\nFormato de X: {X_treino.shape}  |  Formato de Y: {Y_treino.shape}")

## 5. Treinamento do modelo (Random Forest multi-output)

Com `X` e `Y` já separados em treino, teste e verificação, o próximo passo é
treinar o modelo. `RandomForestRegressor` suporta múltiplas saídas nativamente:
cada folha armazena um vetor de médias, uma por termopar.

O número de folds é `min(N_FOLDS_DESEJADO, linhas de treino)`, com mínimo de 2.
Antes esse valor era fixo em 3, o que quebrava quando o conjunto de treino ficava
menor que isso. Com o filtro que remove termopares de baixa cobertura antes da
divisão treino/teste, esse caso deixa de ocorrer nas sete áreas atuais, mas o
limite protege contra planilhas menores no futuro.

In [ ]:
# ============================================================
# CÉLULA 5: TREINAMENTO DO MODELO (RANDOM FOREST MULTI-OUTPUT)
# RandomForestRegressor suporta múltiplas saídas nativamente:
# cada folha armazena um vetor de médias, uma por termopar.
# ============================================================
GRADE_HIPERPARAMETROS = {
    "n_estimators":     [100, 300, 500, 800],
    "max_depth":        [3, 5, 8, 12, 20, None],
    "min_samples_leaf": [1, 2],
    "max_features":     ["sqrt", None],
}

# Nunca pedir mais folds do que há linhas de treino.
n_folds = max(2, min(N_FOLDS_DESEJADO, len(X_treino)))
if n_folds < N_FOLDS_DESEJADO:
    print(f"Aviso: {len(X_treino)} linhas de treino, usando {n_folds} folds "
          f"em vez de {N_FOLDS_DESEJADO}.\n")

busca = GridSearchCV(
    estimator=RandomForestRegressor(random_state=SEMENTE, n_jobs=-1),
    param_grid=GRADE_HIPERPARAMETROS,
    cv=n_folds,                 # poucas linhas: 3 folds em vez de 5
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
)
busca.fit(X_treino, Y_treino)

modelo = busca.best_estimator_

print("Melhores hiperparâmetros:")
for parametro, valor in busca.best_params_.items():
    print(f"   {parametro}: {valor}")

print(f"\nSaídas do modelo: {modelo.n_outputs_} termopares")

# A importância é única para o modelo: reflete todos os alvos em conjunto.
importancia = (
    pd.Series(modelo.feature_importances_, index=VARIAVEIS_INDEPENDENTES)
      .sort_values(ascending=False)
)
print(f"\nImportância das variáveis:\n{importancia.round(4).to_string()}")

plt.figure(figsize=(9, 5))
plt.barh(importancia.index[::-1], importancia.values[::-1], color="#4C72B0")
plt.xlabel("Importância")
plt.title(f"Importância das variáveis, {CAMINHO_PLANILHA.split('/')[-1]}",
          fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Avaliação estatística por termopar

MAE, MSE, RMSE, R² e MAPE calculados coluna a coluna nos três conjuntos, seguidos
da média entre termopares e da fração de previsões dentro da margem de 10%
acordada com o parceiro.

In [ ]:
# ============================================================
# CÉLULA 6: AVALIAÇÃO ESTATÍSTICA POR TERMOPAR
# ============================================================
def avaliar_multioutput(Y_real, Y_previsto, nome_conjunto, alvos):
    """Calcula métricas para cada coluna-alvo separadamente."""
    linhas = []
    for indice, termopar in enumerate(alvos):
        real = Y_real.iloc[:, indice].values
        previsto = Y_previsto[:, indice]
        mse = mean_squared_error(real, previsto)
        linhas.append({
            "conjunto": nome_conjunto,
            "termopar": termopar,
            "MAE":      mean_absolute_error(real, previsto),
            "MSE":      mse,
            "RMSE":     np.sqrt(mse),
            "R2":       r2_score(real, previsto),
            "MAPE_%":   np.mean(np.abs((real - previsto) / real)) * 100,
        })
    return linhas


previsoes_treino      = modelo.predict(X_treino)
previsoes_teste       = modelo.predict(X_teste)
previsoes_verificacao = modelo.predict(X_verificacao)

avaliacao = pd.DataFrame(
    avaliar_multioutput(Y_treino,      previsoes_treino,      "Treino",      VARIAVEIS_ALVO)
  + avaliar_multioutput(Y_teste,       previsoes_teste,       "Teste",       VARIAVEIS_ALVO)
  + avaliar_multioutput(Y_verificacao, previsoes_verificacao, "Verificação", VARIAVEIS_ALVO)
)

print("=" * 100)
print(f"DESEMPENHO POR TERMOPAR: {CAMINHO_PLANILHA}")
print("=" * 100)
print(avaliacao.round(4).to_string(index=False))

print("\n" + "=" * 100)
print("MÉDIA ENTRE OS TERMOPARES")
print("=" * 100)
print(
    avaliacao.groupby("conjunto", sort=False)[["MAE", "MSE", "RMSE", "R2", "MAPE_%"]]
             .mean().round(4).to_string()
)

erro_percentual_teste = np.abs((Y_teste.values - previsoes_teste) / Y_teste.values)
print(f"\nPrevisões dentro da margem de 10% (teste, todos os termopares): "
      f"{(erro_percentual_teste <= 0.10).mean():.1%}")

## 7. Execução em lote: as sete áreas

As seções anteriores tratam a leitura da planilha, a seleção de alvos, a
separação treino/teste/verificação e o treinamento do Random Forest para uma
única área por vez. Esta seção encadeia essas mesmas funções em um único
pipeline e roda as sete planilhas em sequência, para conferir de uma vez que
nenhuma delas quebra e comparar o desempenho entre áreas. É opcional: o fluxo
de uma área por vez continua sendo o caminho para analisar uma área em
detalhe.

In [ ]:
# ============================================================
# CÉLULA 7: EXECUÇÃO EM LOTE NAS SETE PLANILHAS
# ============================================================
PLANILHAS = [
    "../data/mainback_data.csv",
    "../data/coil_temps_data.csv",
    "../data/main_vent_sys_data.csv",
    "../data/ct_knobs_data.csv",
    "../data/ipc_comp_data.csv",
    "../data/ipc_cooling_data.csv",
    "../data/oven_hmi_data.csv",
]

# Guarda, por área, os insumos crus (conjuntos, previsões, modelo e a própria
# busca em grade) usados nas seções seguintes de sobreajuste em lote e de
# tuning: a célula de métricas do Artefato 6 já sabe calcular MAE/RMSE/R² e o
# gap treino-teste por termopar, e a seção de tuning lê o `cv_results_` da
# busca. Guardar aqui evita refazer a busca em grade depois.
INSUMOS_LOTE = []


def treinar_area(caminho, cobertura_minima=COBERTURA_MINIMA_ALVO, n_folds_desejado=N_FOLDS_DESEJADO):
    """Pipeline completo de uma planilha: leitura, tratamento, treino e avaliação."""
    df = pd.read_csv(caminho)
    alvos, _ = identificar_alvos(df, VARIAVEIS_INDEPENDENTES)
    alvos, removidos = filtrar_alvos_por_cobertura(df, alvos, cobertura_minima)
    if not alvos:
        raise ValueError(f"{caminho}: nenhum termopar com cobertura >= {cobertura_minima:.0%}")

    tratado = df.dropna(subset=alvos)
    tratado = padronizar_categorias(tratado, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES)
    X_area, _ = codificar_ordinal(tratado, VARIAVEIS_INDEPENDENTES)
    Y_area = tratado[alvos]

    X_tr, X_resto, Y_tr, Y_resto = train_test_split(
        X_area, Y_area, train_size=PROPORCAO_TREINO, random_state=SEMENTE)
    X_te, X_ve, Y_te, Y_ve = train_test_split(
        X_resto, Y_resto,
        test_size=PROPORCAO_VERIFICACAO / (PROPORCAO_TESTE + PROPORCAO_VERIFICACAO),
        random_state=SEMENTE)

    n_folds_area = max(2, min(n_folds_desejado, len(X_tr)))
    busca_area = GridSearchCV(
        estimator=RandomForestRegressor(random_state=SEMENTE, n_jobs=-1),
        param_grid=GRADE_HIPERPARAMETROS,
        cv=n_folds_area,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    ).fit(X_tr, Y_tr)

    modelo_area = busca_area.best_estimator_
    previsoes_treino_area = modelo_area.predict(X_tr)
    previsoes_teste_area = modelo_area.predict(X_te)

    nome_area = caminho.split("/")[-1]
    INSUMOS_LOTE.append({
        "nome_area": nome_area,
        "X_tr": X_tr,
        "X_te": X_te,
        "Y_tr": Y_tr,
        "Y_te": Y_te,
        "previsoes_treino": previsoes_treino_area,
        "previsoes_teste": previsoes_teste_area,
        "alvos": alvos,
        "modelo": modelo_area,
        "busca": busca_area,
        "n_folds": n_folds_area,
    })

    metricas_teste = pd.DataFrame(
        avaliar_multioutput(Y_te, previsoes_teste_area, "Teste", alvos))
    erro_pct = np.abs((Y_te.values - previsoes_teste_area) / Y_te.values)

    return {
        "planilha": nome_area,
        "linhas": len(tratado),
        "alvos": len(alvos),
        "alvos_removidos": len(removidos),
        "MAE_teste": metricas_teste["MAE"].mean(),
        "RMSE_teste": metricas_teste["RMSE"].mean(),
        "R2_teste": metricas_teste["R2"].mean(),
        "MAPE_teste_%": metricas_teste["MAPE_%"].mean(),
        "dentro_de_10%": (erro_pct <= 0.10).mean(),
    }


resultados = []
for caminho in PLANILHAS:
    try:
        resultados.append(treinar_area(caminho))
        print(f"OK   {caminho}")
    except Exception as erro:
        print(f"FALHA {caminho}: {type(erro).__name__}: {erro}")

comparativo = pd.DataFrame(resultados)
print("\n" + "=" * 110)
print("COMPARATIVO ENTRE ÁREAS: métricas médias no conjunto de teste")
print("=" * 110)
print(comparativo.round(4).to_string(index=False))

## 8. Métricas de avaliação do modelo candidato: Artefato 6, item b)

Esta célula implementa as três métricas apresentadas na Seção 4.3 da documentação:
MAE, RMSE e R², sempre por termopar (`tc_name`) e sobre o conjunto de teste
(`Y_teste`, `previsoes_teste`), como descrito ali.

Ela difere da avaliação exploratória anterior (MAE, MSE, RMSE, R² e MAPE por
termopar) em três pontos, todos alinhados aos riscos já registrados na Seção
4.1.5:

- reporta o R² **mediano** entre termopares, e não a média: uma média de razões
  afunda com um único alvo catastrófico e esconde problemas pontuais;
- inclui um baseline `DummyRegressor(strategy="mean")` ajustado no treino, para
  que MAE e RMSE sejam lidos contra uma referência realista, e não isoladamente
  (R03);
- calcula o afastamento entre o MAE de treino e o de teste, indicador de
  sobreajuste, risco elevado aqui pelo número reduzido de execuções disponíveis
  (R02).

A célula é independente da avaliação exploratória: detecta sozinha se está
rodando logo após o fluxo de área única ou após a execução em lote (que popula
`TABELAS_METRICAS`), sem exigir nenhuma alteração de código para trocar de
contexto.

In [ ]:

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -----------------------------------------------------------------------------
# CONFIGURAÇÃO: único bloco que pode precisar de ajuste.
# Deixe em None para detecção automática; preencha só se a detecção errar.
# -----------------------------------------------------------------------------
MODELO_CANDIDATO = None    # ex.: "Random Forest"
AREA = None                # ex.: "IPC Comp"


# =============================================================================
# 1. IDENTIFICAÇÃO DO ALGORITMO
# =============================================================================

_TRADUCAO_MODELOS = {
    "RandomForestRegressor": "Random Forest",
    "DecisionTreeRegressor": "Árvore de Decisão",
    "GradientBoostingRegressor": "Gradient Boosting",
}


def _classe_do_estimador(obj):
    """Nome da classe do modelo, desembrulhando os wrappers mais comuns."""
    est = getattr(obj, "best_estimator_", obj)
    if type(est).__name__.startswith(("MultiOutput", "RegressorChain")):
        est = getattr(est, "estimator", est)
    return type(est).__name__


def _nome_legivel(obj):
    return _TRADUCAO_MODELOS.get(_classe_do_estimador(obj), _classe_do_estimador(obj))


# =============================================================================
# 2. VALIDAÇÃO DAS ENTRADAS
# =============================================================================

def _alinhar_colunas(y, nomes_alvos, rotulo):
    """Garante que as colunas de um DataFrame sigam a ordem da lista de alvos."""
    if not isinstance(y, pd.DataFrame):
        return y

    colunas, alvos = list(y.columns), list(nomes_alvos)
    if colunas == alvos:
        return y
    if set(colunas) == set(alvos):
        print(f"[aviso] {rotulo}: colunas fora da ordem da lista de alvos. "
              "Reordenadas automaticamente.")
        return y[alvos]

    faltando = [a for a in alvos if a not in colunas]
    sobrando = [c for c in colunas if c not in alvos]
    raise ValueError(
        f"{rotulo}: as colunas não correspondem à lista de alvos.\n"
        f"  ausentes em {rotulo}: {faltando[:5]}\n"
        f"  fora da lista de alvos: {sobrando[:5]}\n"
        "Sem essa correspondência as métricas seriam atribuídas ao termopar errado."
    )


def _validar_formato(y_treino, y_teste, pred_teste, nomes_alvos, pred_treino):
    """Confere número de colunas e de linhas antes de calcular qualquer coisa."""
    n_alvos = len(nomes_alvos)

    for nome, arr in (("Y_treino", y_treino), ("Y_teste", y_teste),
                      ("previsoes_teste", pred_teste)):
        n_col = 1 if arr.ndim == 1 else arr.shape[1]
        if n_col != n_alvos:
            raise ValueError(
                f"{nome} tem {n_col} coluna(s), mas a lista de alvos tem "
                f"{n_alvos} termopares. Verifique se está passando a lista "
                "certa (VARIAVEIS_ALVO na seção de área única, `alvos` no lote)."
            )

    if len(y_teste) != len(pred_teste):
        raise ValueError(
            f"Y_teste tem {len(y_teste)} linhas e previsoes_teste tem "
            f"{len(pred_teste)}. As previsões não são deste conjunto."
        )
    if pred_treino is not None and len(y_treino) != len(pred_treino):
        raise ValueError(
            f"Y_treino tem {len(y_treino)} linhas e previsoes_treino tem "
            f"{len(pred_treino)}. As previsões não são deste conjunto."
        )


# =============================================================================
# 3. MÉTRICAS POR TERMOPAR
# =============================================================================

def metricas_por_alvo(y_treino, y_teste, pred_teste, nomes_alvos, key_area,
                      modelo=None, pred_treino=None):
    """MAE, RMSE e R² de cada termopar, com baseline de média.

    As métricas são medidas por termopar, e não sobre o conjunto agregado,
    porque o R² só significa algo em relação ao preditor ingênuo com que
    compara: agregado, esse preditor seria a média de todos os sensores do
    produto, valor que nenhum engenheiro usaria como estimativa. Como as regiões
    operam em patamares térmicos distintos, o R² agregado mediria sobretudo essa
    diferença de patamar, já conhecida, e não o que o modelo aprendeu.

    `pred_treino` é opcional. Quando fornecido, a tabela ganha o MAE de treino e
    o afastamento até o MAE de teste, indicador de sobreajuste.
    """
    # Alinha colunas quando os conjuntos vêm como DataFrame, antes de virarem
    # matrizes anônimas, depois do asarray o nome da coluna se perde.
    y_treino = _alinhar_colunas(y_treino, nomes_alvos, f"Y_treino ({key_area})")
    y_teste = _alinhar_colunas(y_teste, nomes_alvos, f"Y_teste ({key_area})")
    pred_teste = _alinhar_colunas(pred_teste, nomes_alvos, f"previsoes_teste ({key_area})")
    if pred_treino is not None:
        pred_treino = _alinhar_colunas(pred_treino, nomes_alvos,
                                       f"previsoes_treino ({key_area})")

    y_treino = np.asarray(y_treino, dtype=float)
    y_teste = np.asarray(y_teste, dtype=float)
    pred_teste = np.asarray(pred_teste, dtype=float)
    if pred_treino is not None:
        pred_treino = np.asarray(pred_treino, dtype=float)

    _validar_formato(y_treino, y_teste, pred_teste, nomes_alvos, pred_treino)

    if y_treino.ndim == 1:
        y_treino, y_teste = y_treino.reshape(-1, 1), y_teste.reshape(-1, 1)
        pred_teste = pred_teste.reshape(-1, 1)
        if pred_treino is not None:
            pred_treino = pred_treino.reshape(-1, 1)

    linhas = []
    for j, alvo in enumerate(nomes_alvos):
        real, prev, treino = y_teste[:, j], pred_teste[:, j], y_treino[:, j]

        # Célula vazia na planilha original = TC não medido naquele run.
        # Não é erro do modelo, é ausência de dado: fica fora da métrica.
        ok = ~np.isnan(real) & ~np.isnan(prev)
        ok_treino = ~np.isnan(treino)
        n = int(ok.sum())
        if n == 0 or ok_treino.sum() == 0:
            continue

        real_ok, prev_ok = real[ok], prev[ok]

        # Baseline realista: média daquele termopar no TREINO.
        dummy = DummyRegressor(strategy="mean")
        dummy.fit(np.zeros((int(ok_treino.sum()), 1)), treino[ok_treino])
        prev_base = dummy.predict(np.zeros((n, 1)))

        mae = mean_absolute_error(real_ok, prev_ok)
        rmse = float(np.sqrt(mean_squared_error(real_ok, prev_ok)))
        mae_base = mean_absolute_error(real_ok, prev_base)

        # R² exige variância no teste: com n < 2 ou alvo constante o denominador
        # é zero e a métrica não existe. Esse denominador usa a média do próprio
        # conjunto de teste, e não a do treino usada pelo baseline acima.
        r2 = r2_score(real_ok, prev_ok) if n >= 2 and np.ptp(real_ok) > 0 else np.nan

        registro = {
            "modelo": _nome_legivel(modelo) if modelo is not None else None,
            "key_area": key_area,
            "tc_name": alvo,
            "n_teste": n,
            "MAE_C": mae,
            "RMSE_C": rmse,
            "R2": r2,
            "razao_RMSE_MAE": rmse / mae if mae > 0 else np.nan,
            "MAE_baseline_treino_C": mae_base,
            "ganho_sobre_baseline_%": (1 - mae / mae_base) * 100 if mae_base > 0 else np.nan,
        }

        if pred_treino is not None:
            ok_tr = ok_treino & ~np.isnan(pred_treino[:, j])
            if ok_tr.sum():
                mae_tr = mean_absolute_error(treino[ok_tr], pred_treino[ok_tr, j])
                registro["MAE_treino_C"] = mae_tr
                registro["gap_C"] = mae - mae_tr
                registro["gap_relativo_%"] = (
                    (mae - mae_tr) / mae_tr * 100 if mae_tr > 0 else np.nan
                )

        linhas.append(registro)

    return pd.DataFrame(linhas)


# =============================================================================
# 4. AGREGAÇÃO POR key_area
# =============================================================================

def resumo_por_area(tabela_alvos):
    """Agrega as métricas por região do produto.

    Reporta a CONTAGEM de termopares com R² < 0, e não só a média do R²: uma
    média de razões afunda com um único alvo catastrófico e não distingue um
    problema generalizado de um problema pontual.
    """
    tem_gap = "gap_C" in tabela_alvos.columns

    linhas = []
    for area, g in tabela_alvos.groupby("key_area"):
        r2 = g["R2"].dropna()
        mae, rmse = g["MAE_C"].mean(), g["RMSE_C"].mean()
        mae_base = g["MAE_baseline_treino_C"].mean()
        registro = {
            "key_area": area,
            "n_alvos": len(g),
            "n_teste_medio": g["n_teste"].mean(),
            "MAE_C": mae,
            "RMSE_C": rmse,
            "razao_RMSE_MAE": rmse / mae if mae > 0 else np.nan,
            "R2_mediano": r2.median() if len(r2) else np.nan,
            "alvos_R2_negativo": int((r2 < 0).sum()),
            "pct_R2_negativo": (r2 < 0).mean() * 100 if len(r2) else np.nan,
            "MAE_baseline_treino_C": mae_base,
            "ganho_sobre_baseline_%": (1 - mae / mae_base) * 100 if mae_base > 0 else np.nan,
        }
        if tem_gap:
            registro["MAE_treino_C"] = g["MAE_treino_C"].mean()
            registro["gap_C"] = g["gap_C"].mean()
            registro["gap_relativo_%"] = g["gap_relativo_%"].mean()
        linhas.append(registro)

    return pd.DataFrame(linhas).sort_values("MAE_C").reset_index(drop=True)


# =============================================================================
# 5. LEITURA E EXPORTAÇÃO
# =============================================================================

def imprimir_leitura(resumo, modelo, origem):
    """Traduz as três métricas para linguagem de engenharia, área por área."""
    tem_gap = "gap_C" in resumo.columns

    print("=" * 78)
    print("ARTEFATO 6, item b) METRICAS DO MODELO CANDIDATO")
    print(f"Modelo: {modelo}   |   Conjunto: TESTE   |   Origem: {origem}")
    print("=" * 78)

    for _, r in resumo.iterrows():
        print(f"\n{r['key_area']}  ({int(r['n_alvos'])} termopares, "
              f"n_teste medio = {r['n_teste_medio']:.1f})")
        print(f"  MAE  = {r['MAE_C']:6.2f} °C")
        print(f"  RMSE = {r['RMSE_C']:6.2f} °C   (razao RMSE/MAE = "
              f"{r['razao_RMSE_MAE']:.2f}"
              f"{'  <- cauda de erros grandes' if r['razao_RMSE_MAE'] > 1.4 else ''})")
        print(f"  R2   = {r['R2_mediano']:6.3f} (mediano; referencia = media do "
              f"proprio conjunto de teste)")
        print(f"         {int(r['alvos_R2_negativo'])} de {int(r['n_alvos'])} "
              f"termopares com R2 < 0 ({r['pct_R2_negativo']:.0f}%)")
        print(f"  Baseline realista (media do treino): MAE = "
              f"{r['MAE_baseline_treino_C']:6.2f} °C  ->  modelo "
              f"{r['ganho_sobre_baseline_%']:+5.1f}% melhor")
        if tem_gap and pd.notna(r["gap_C"]):
            print(f"  Sobreajuste: MAE treino {r['MAE_treino_C']:.2f} °C -> teste "
                  f"{r['MAE_C']:.2f} °C  ({r['gap_C']:+.2f} °C, "
                  f"{r['gap_relativo_%']:+.1f}% relativo)")
        if r["alvos_R2_negativo"] > r["n_alvos"] / 2:
            print("  ATENCAO: na maioria dos termopares desta area o modelo "
                  "perde para a previsao da media.")
        if r["n_teste_medio"] < 5:
            print(f"  ATENCAO: media de {r['n_teste_medio']:.1f} leituras por "
                  "termopar no teste. O R2 e instavel nesse tamanho de amostra "
                  "e deve ser lido como indicativo, nao como medicao.")

    if tem_gap:
        print("\nNota sobre o afastamento treino -> teste: o R03 fixa o gatilho em")
        print("\"cinco pontos percentuais\", mas nao define a grandeza sobre a qual o")
        print("percentual incide: a mesma ambiguidade de unidade registrada em R06")
        print("para o criterio de 10%. Por isso o afastamento e reportado em °C e em")
        print("variacao relativa, sem converter um no outro.")
    print("\n" + "=" * 78)


def tabela_markdown(resumo):
    """Tabela em markdown com os resultados obtidos.

    O item b) do artefato documenta a definição das métricas; os valores
    alcançados pelo modelo candidato pertencem ao item c). A tabela é gerada
    aqui para que notebook e documentação não divirjam quando o item c) for
    escrito.
    """
    def v(x, casas=2):
        return "n/d" if pd.isna(x) else f"{x:.{casas}f}".replace(".", ",")

    linhas = [
        "| `key_area` | n alvos | n teste | MAE (°C) | RMSE (°C) | RMSE/MAE | "
        "R² mediano | Alvos com R² < 0 | MAE baseline (°C) |",
        "|---|---|---|---|---|---|---|---|---|",
    ]
    for _, r in resumo.iterrows():
        linhas.append(
            f"| {r['key_area']} | {int(r['n_alvos'])} | {r['n_teste_medio']:.0f} | "
            f"{v(r['MAE_C'])} | {v(r['RMSE_C'])} | {v(r['razao_RMSE_MAE'])} | "
            f"{v(r['R2_mediano'], 3)} | {int(r['alvos_R2_negativo'])} de "
            f"{int(r['n_alvos'])} | {v(r['MAE_baseline_treino_C'])} |"
        )
    return "\n".join(linhas)


# =============================================================================
# 6. DETECÇÃO DO CONTEXTO: qual notebook, qual seção
# =============================================================================

def _nome_do_modelo(ns):
    """Descobre o algoritmo pelo objeto treinado que estiver no namespace."""
    encontrados = [
        _classe_do_estimador(obj)
        for nome, obj in list(ns.items())
        if not nome.startswith("_") and isinstance(obj, BaseEstimator)
        and (hasattr(obj, "predict") or hasattr(obj, "best_estimator_"))
    ]
    conhecidos = [c for c in encontrados if c in _TRADUCAO_MODELOS]
    if len(set(conhecidos)) > 1:
        print(f"[aviso] Mais de um algoritmo em memória: {sorted(set(conhecidos))}. "
              "Preencha MODELO_CANDIDATO se o nome abaixo estiver errado.")
    if conhecidos:
        return _TRADUCAO_MODELOS[conhecidos[0]]
    return encontrados[-1] if encontrados else None


def _nome_da_area(ns):
    for chave in ("AREA", "area", "KEY_AREA", "key_area", "nome_area", "setor", "SETOR"):
        valor = ns.get(chave)
        if isinstance(valor, str) and valor.strip():
            return valor.strip()
    return "área única"


def executar_metricas():
    """Ponto de entrada. Descobre sozinho a seção do notebook e roda tudo."""
    ns = globals()

    # Caminho A: seção em lote, várias key_area de uma vez.
    if isinstance(ns.get("TABELAS_METRICAS"), list) and ns["TABELAS_METRICAS"]:
        metricas_alvo = pd.concat(ns["TABELAS_METRICAS"], ignore_index=True)
        origem = f"lote ({metricas_alvo['key_area'].nunique()} key_area)"

    # Caminho B: uma área por vez. Não exige alterar célula nenhuma.
    elif all(k in ns for k in ("Y_treino", "Y_teste", "previsoes_teste", "VARIAVEIS_ALVO")):
        area = AREA or _nome_da_area(ns)
        modelo_obj = next(
            (o for n, o in ns.items()
             if not n.startswith("_") and isinstance(o, BaseEstimator)
             and _classe_do_estimador(o) in _TRADUCAO_MODELOS),
            None,
        )
        metricas_alvo = metricas_por_alvo(
            ns["Y_treino"], ns["Y_teste"], ns["previsoes_teste"],
            ns["VARIAVEIS_ALVO"], area,
            modelo=modelo_obj,
            pred_treino=ns.get("previsoes_treino"),   # habilita o gap
        )
        origem = f"área única ({area})"

    else:
        raise RuntimeError(
            "Nenhum conjunto de previsões encontrado em memória.\n"
            "  - Uma área por vez: rode antes a célula que gera `Y_treino`, "
            "`Y_teste`, `previsoes_teste` e `VARIAVEIS_ALVO`.\n"
            "  - Lote: acrescente dentro de `treinar_area`, depois do modelo "
            "treinado:\n"
            "        previsoes_tr = modelo_area.predict(X_tr)\n"
            "        previsoes_te = modelo_area.predict(X_te)\n"
            "        TABELAS_METRICAS.append(metricas_por_alvo(\n"
            "            Y_tr, Y_te, previsoes_te, alvos, area,\n"
            "            modelo=modelo_area, pred_treino=previsoes_tr))\n"
            "    e declare `TABELAS_METRICAS = []` antes de rodar o lote."
        )

    # Nome do algoritmo, em ordem de confiança: preenchido à mão, registrado na
    # própria chamada, e só então varredura do namespace (que não alcança
    # modelos locais a uma função, como no lote).
    registrado = metricas_alvo["modelo"].dropna().unique() if "modelo" in metricas_alvo else []
    modelo = (MODELO_CANDIDATO
              or (registrado[0] if len(registrado) == 1 else None)
              or _nome_do_modelo(ns)
              or "preencher MODELO_CANDIDATO")
    if len(registrado) > 1:
        print(f"[aviso] A tabela mistura algoritmos: {sorted(registrado)}. "
              "O Artefato 6 pede um único modelo candidato.")

    resumo = resumo_por_area(metricas_alvo)
    imprimir_leitura(resumo, modelo, origem)

    print("\nTabela de resultados em markdown (para o item c do artefato):\n")
    print(tabela_markdown(resumo))

    print("\n\n10 termopares de pior R² (onde o modelo perde para a média):\n")
    print(metricas_alvo.sort_values("R2").head(10)
          [["key_area", "tc_name", "n_teste", "MAE_C", "RMSE_C", "R2"]]
          .round(2).to_string(index=False))

    return metricas_alvo, resumo


# =============================================================================
# 7. EXECUÇÃO
# =============================================================================

metricas_alvo, resumo = executar_metricas()

try:
    display(resumo.round(3))
except NameError:          # fora do Jupyter/Colab o display nao existe
    print(resumo.round(3).to_string(index=False))

### Leitura do resultado acima

Nesta execução (`main_vent_sys_data.csv`), o modelo erra em média 4,21 °C no
teste (MAE), com RMSE de 6,43 °C, razão RMSE/MAE de 1,53, sinal de que parte do
erro se concentra em poucos termopares específicos, e não distribuído igualmente
entre os cinco alvos. O R² mediano de 0,329 indica que o modelo aprendeu parte
da relação entre configuração e temperatura, mas está longe de explicar toda a
variação; nenhum dos cinco termopares ficou com R² negativo, ou seja, nenhum
perdeu para a previsão ingênua da média.

O afastamento entre o MAE de treino (3,43 °C) e o de teste (4,21 °C) é de +36,4%
relativo, acima do gatilho de cinco pontos percentuais do R03. Mas, como a nota
impressa acima registra, o R03 não define a grandeza sobre a qual esse percentual
incide, então o valor é reportado aqui para registro, não como critério de
aprovação automática.

Vale a ressalva já impressa pela função `imprimir_leitura`: com média de 11
leituras de teste por termopar, o R² é instável nesse tamanho de amostra e deve
ser lido como indicativo. Os valores desta execução ilustram a definição das
métricas (item b); a leitura oficial do modelo candidato, sobre todas as sete
áreas, pertence ao item c) do Artefato 6.

## 9. Sobreajuste em lote: treino vs. teste nas sete áreas

A célula de métricas do Artefato 6 (item b) já calcula, por termopar, o
afastamento entre o MAE de treino e o de teste (`gap_C`, `gap_relativo_%`)
quando recebe `pred_treino`. Mas ela só faz essa conta para a área única
carregada no início do notebook (hoje `main_vent_sys_data.csv`).

Esta célula reaproveita exatamente as mesmas funções (`metricas_por_alvo`,
`resumo_por_area`, `imprimir_leitura`, já definidas ali) sobre os insumos
guardados pela execução em lote (`INSUMOS_LOTE`), para obter o mesmo relatório
de sobreajuste, desta vez agregado nas sete `key_area` de uma vez: o retrato
completo do risco R03 (Seção 4.1.5) para o Random Forest, e não apenas de uma
área isolada.

Requer ter rodado antes a execução em lote (que popula `INSUMOS_LOTE`) e a
célula de métricas do Artefato 6 (que define as funções reutilizadas aqui).

In [ ]:
# ============================================================
# CÉLULA 9: SOBREAJUSTE EM LOTE (TREINO x TESTE, SETE ÁREAS)
# ============================================================
TABELAS_METRICAS = [
    metricas_por_alvo(
        insumo["Y_tr"], insumo["Y_te"], insumo["previsoes_teste"],
        insumo["alvos"], insumo["nome_area"],
        modelo=insumo["modelo"], pred_treino=insumo["previsoes_treino"],
    )
    for insumo in INSUMOS_LOTE
]

metricas_alvo_lote, resumo_lote = executar_metricas()

try:
    display(resumo_lote.round(3))
except NameError:          # fora do Jupyter/Colab o display nao existe
    print(resumo_lote.round(3).to_string(index=False))

print("\n\n10 termopares de MAIOR sobreajuste (maior gap_relativo_% treino -> teste):\n")
print(
    metricas_alvo_lote.sort_values("gap_relativo_%", ascending=False).head(10)
    [["key_area", "tc_name", "MAE_treino_C", "MAE_C", "gap_C", "gap_relativo_%"]]
    .round(2).to_string(index=False)
)

## 10. Tuning de hiperparâmetros por Grid Search: registro da busca

As seções anteriores já usam `GridSearchCV`, mas guardam apenas a combinação
vencedora: o notebook não registrava *o que foi testado*, *por quanto a
vencedora ganhou* nem *se o tuning melhorou alguma coisa*. Esta seção fecha esse
registro, sem repetir a busca: ela lê o `cv_results_` das buscas já executadas
na execução em lote (que por isso passou a guardar o objeto `busca` de cada
área).

**Como o Grid Search funciona aqui.** A grade cruza quatro hiperparâmetros
(`n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`), somando 96
combinações. Cada uma é treinada `k` vezes em validação cruzada *dentro do
conjunto de treino* (`k = min(3, linhas de treino)`, com piso de 2) e avaliada
pelo MAE médio entre os folds. Vence a combinação de menor MAE, que é então
re-treinada em todo o treino (`best_estimator_`). Teste e verificação não
participam da busca, então o tuning não contamina a avaliação final. A busca é
independente por `key_area`: são sete buscas, 672 combinações avaliadas no
total.

**Por que a grade foi ampliada.** A primeira rodada (`n_estimators` em [100, 300],
`max_depth` em [3, 5, None]) mostrou a maioria das combinações empatando dentro do
ruído entre folds, e a vencedora generalizando pior que o modelo sem tuning em 6 das 7
áreas. Antes de aceitar essa conclusão, a grade foi ampliada para `n_estimators` em
[100, 300, 500, 800] e `max_depth` em [3, 5, 8, 12, 20, None] (mais árvores e mais
valores intermediários de profundidade entre 5 e sem limite), para verificar se o
problema era a grade estreita ou o tamanho da base de treino, como já era esperado.

**O que esta célula acrescenta, além do vencedor:**

- **O tuning valeu a pena?** Compara, no conjunto de teste, a combinação
  escolhida contra o `RandomForestRegressor` sem tuning nenhum (padrões do
  sklearn).
- **A escolha é confiável?** Reporta a margem da vencedora sobre a 2ª colocada
  e a compara com o desvio-padrão entre os folds. Quando a margem cabe dentro
  do ruído da validação, a vencedora é uma entre várias combinações
  equivalentes.
- **Quanto os hiperparâmetros importam?** A curva de sensibilidade mostra o
  quanto cada combinação é pior que a melhor da própria área. Curva achatada
  significa que o Random Forest é pouco sensível ao ajuste naquela área;
  íngreme, que o tuning é decisivo.
- **A grade inteira** da área carregada no início do notebook
  (`CAMINHO_PLANILHA`), para que fique registrada ao menos uma busca completa,
  e não apenas o vencedor.

In [ ]:
# ============================================================
# CÉLULA 10: REGISTRO DO TUNING DE HIPERPARÂMETROS (GRID SEARCH)
# Lê o `cv_results_` das buscas já feitas na execução em lote: nada é re-treinado
# aqui, exceto os modelos-padrão usados como referência de comparação.
# ============================================================
N_COMBINACOES = int(np.prod([len(v) for v in GRADE_HIPERPARAMETROS.values()]))

print("=" * 104)
print("TUNING DE HIPERPARÂMETROS: GRID SEARCH (busca exaustiva em grade)")
print("=" * 104)
print(f"\nGrade testada ({N_COMBINACOES} combinações por área, "
      f"{len(INSUMOS_LOTE)} áreas = {N_COMBINACOES * len(INSUMOS_LOTE)} combinações avaliadas):")
for parametro, valores in GRADE_HIPERPARAMETROS.items():
    print(f"   {parametro:18s} {valores}")
print(f"\nCritério de escolha: menor MAE médio na validação cruzada do conjunto de TREINO")
print(f"Validação cruzada:   k = min({N_FOLDS_DESEJADO}, linhas de treino), mínimo 2 folds")
print("Os conjuntos de teste e verificação não participam da busca.")


def registrar_busca(insumo):
    """Extrai da busca em grade o vencedor, a margem sobre o 2º e a estabilidade."""
    cv = pd.DataFrame(insumo["busca"].cv_results_).sort_values("rank_test_score")
    melhor, segunda = cv.iloc[0], cv.iloc[1]

    # Os hiperparâmetros vêm de `best_params_`, e não das colunas `param_*` do
    # cv_results_: lá, `None` chega como valor mascarado e vira NaN na tabela,
    # sumindo das contagens por valor mais adiante.
    vencedores = insumo["busca"].best_params_

    mae_melhor = -melhor["mean_test_score"]
    mae_segunda = -segunda["mean_test_score"]
    desvio_melhor = melhor["std_test_score"]

    # Combinações estatisticamente empatadas: dentro de 1 desvio-padrão entre
    # folds do melhor MAE. Quanto mais empatadas, menos a escolha da vencedora
    # significa: a diferença entre elas cabe dentro do ruído da validação.
    empatadas = int(((-cv["mean_test_score"]) <= mae_melhor + desvio_melhor).sum())

    # Referência: o mesmo Random Forest sem tuning nenhum (padrões do sklearn).
    padrao = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1)
    padrao.fit(insumo["X_tr"], insumo["Y_tr"])
    pred_teste_padrao = padrao.predict(insumo["X_te"])
    mae_teste_padrao = mean_absolute_error(insumo["Y_te"], pred_teste_padrao)
    mae_teste_ajustado = mean_absolute_error(insumo["Y_te"], insumo["previsoes_teste"])
    r2_teste_padrao = r2_score(insumo["Y_te"], pred_teste_padrao)
    r2_teste_ajustado = r2_score(insumo["Y_te"], insumo["previsoes_teste"])
    mape_teste_padrao = np.mean(np.abs(
        (insumo["Y_te"].values - pred_teste_padrao) / insumo["Y_te"].values)) * 100
    mape_teste_ajustado = np.mean(np.abs(
        (insumo["Y_te"].values - insumo["previsoes_teste"]) / insumo["Y_te"].values)) * 100

    return {
        "key_area": insumo["nome_area"],
        "n_treino": len(insumo["X_tr"]),
        "folds": insumo["n_folds"],
        "n_estimators": vencedores["n_estimators"],
        "max_depth": str(vencedores["max_depth"]),
        "min_samples_leaf": vencedores["min_samples_leaf"],
        "max_features": str(vencedores["max_features"]),
        "MAE_cv_melhor": mae_melhor,
        "desvio_entre_folds": desvio_melhor,
        "margem_sobre_2o_C": mae_segunda - mae_melhor,
        "combinacoes_empatadas": empatadas,
        "MAE_teste_padrao": mae_teste_padrao,
        "MAE_teste_ajustado": mae_teste_ajustado,
        "ganho_do_tuning_C": mae_teste_padrao - mae_teste_ajustado,
        "R2_teste_padrao": r2_teste_padrao,
        "R2_teste_ajustado": r2_teste_ajustado,
        "MAPE_teste_padrao": mape_teste_padrao,
        "MAPE_teste_ajustado": mape_teste_ajustado,
    }


tuning = pd.DataFrame([registrar_busca(i) for i in INSUMOS_LOTE])

print("\n" + "=" * 104)
print("VENCEDORA DA BUSCA EM CADA ÁREA")
print("=" * 104)
print(
    tuning[["key_area", "n_treino", "folds", "n_estimators", "max_depth",
            "min_samples_leaf", "max_features", "MAE_cv_melhor"]]
    .round(3).to_string(index=False)
)

print("\n" + "=" * 104)
print("O TUNING VALEU A PENA? (conjunto de teste: padrão do sklearn vs. combinação escolhida)")
print("=" * 104)
print(
    tuning[["key_area", "MAE_teste_padrao", "MAE_teste_ajustado", "ganho_do_tuning_C"]]
    .round(3).to_string(index=False)
)
print(f"\nGanho médio do tuning: {tuning['ganho_do_tuning_C'].mean():+.3f} °C "
      f"({(tuning['ganho_do_tuning_C'] > 0).sum()} de {len(tuning)} áreas melhoraram)")

print("\n" + "=" * 104)
print("QUÃO CONFIÁVEL É A ESCOLHA? (margem sobre a 2ª colocada vs. ruído entre folds)")
print("=" * 104)
print(
    tuning[["key_area", "MAE_cv_melhor", "margem_sobre_2o_C", "desvio_entre_folds",
            "combinacoes_empatadas"]]
    .round(3).to_string(index=False)
)
print(f"\nLeitura: `combinacoes_empatadas` conta quantas das {N_COMBINACOES} combinações ficaram")
print("dentro de um desvio-padrão entre folds da vencedora. Quando esse número é alto, a")
print("diferença entre a vencedora e as demais cabe dentro do ruído da validação cruzada:")
print("a combinação escolhida é uma entre várias equivalentes, não uma configuração superior.")

# Diagnóstico automático: cruza as duas tabelas acima. Sem isto, cada uma delas
# sozinha admite leitura benigna; juntas, apontam seleção sobre ruído.
areas_sem_ganho = int((tuning["ganho_do_tuning_C"] <= 0).sum())
areas_empatadas = int((tuning["combinacoes_empatadas"] >= N_COMBINACOES * 0.5).sum())
if areas_sem_ganho > len(tuning) / 2 and areas_empatadas > len(tuning) / 2:
    print(f"\nATENCAO (seleção sobre ruído): em {areas_empatadas} de {len(tuning)} áreas a maioria das")
    print(f"{N_COMBINACOES} combinações empata com a vencedora, e em {areas_sem_ganho} de {len(tuning)} áreas a")
    print("combinação escolhida generalizou PIOR que o Random Forest sem tuning. Com margens")
    print("de centésimos de °C contra desvios entre folds de ordem de grandeza maior, a busca")
    print("está ordenando ruído, não desempenho. A causa é dimensional (16 a 39 linhas de")
    print("treino por área, risco R02 da Seção 4.1.5), não a grade nem o algoritmo: ampliar a")
    print("grade não corrige isso, e trocar Grid Search por Random Search também não.")

# Frequência de cada valor entre as sete vencedoras: base para discutir um
# conjunto único de hiperparâmetros, caso o projeto queira abandonar a busca
# por área no futuro.
print("\n" + "=" * 104)
print("VALORES MAIS ESCOLHIDOS ENTRE AS SETE ÁREAS")
print("=" * 104)
for parametro in GRADE_HIPERPARAMETROS:
    contagem = tuning[parametro].astype(str).value_counts()
    print(f"\n{parametro}:")
    for valor, quantas in contagem.items():
        print(f"   {valor:6s} -> {quantas} de {len(tuning)} áreas")

# Curva de sensibilidade: o quanto cada combinação é pior que a melhor DAQUELA
# área, em %. Curva achatada = a escolha do hiperparâmetro pouco altera o
# resultado; curva íngreme = o tuning é decisivo naquela área.
plt.figure(figsize=(10, 5.5))
for insumo in INSUMOS_LOTE:
    maes = np.sort(-pd.DataFrame(insumo["busca"].cv_results_)["mean_test_score"].values)
    plt.plot(
        range(1, len(maes) + 1),
        (maes / maes[0] - 1) * 100,
        marker="o", markersize=3, linewidth=1.5,
        label=insumo["nome_area"].replace("_data.csv", ""),
    )
plt.xlabel("Ranking da combinacao na busca em grade (1o = vencedora daquela area, ultimo = pior)")
plt.ylabel("MAE de validacao, % acima da vencedora daquela area (0% = a propria vencedora)")
plt.title("Sensibilidade do Random Forest aos hiperparametros, por area\n"
          "(curva achatada = tuning pouco importa; curva ingreme = tuning decisivo)",
          fontsize=12, fontweight="bold")
plt.legend(title="key_area", fontsize=8)
plt.tight_layout()
plt.show()

# Registro completo da grade para a área carregada no início do notebook: as 24 linhas,
# para que o notebook guarde ao menos uma busca inteira, e não só o vencedor.
area_referencia = CAMINHO_PLANILHA.split("/")[-1]
insumo_referencia = next(
    (i for i in INSUMOS_LOTE if i["nome_area"] == area_referencia), INSUMOS_LOTE[0]
)
grade_completa = (
    pd.DataFrame(insumo_referencia["busca"].cv_results_)
    .assign(
        MAE_cv=lambda d: -d["mean_test_score"],
        # `None` chega mascarado do cv_results_; sem isto sairia como NaN.
        param_max_depth=lambda d: d["param_max_depth"].astype(object).where(
            d["param_max_depth"].notna(), "None"),
        param_max_features=lambda d: d["param_max_features"].astype(object).where(
            d["param_max_features"].notna(), "None"),
    )
    .sort_values("MAE_cv")
    [["param_n_estimators", "param_max_depth", "param_min_samples_leaf",
      "param_max_features", "MAE_cv", "std_test_score", "rank_test_score"]]
)

print("\n" + "=" * 104)
print(f"GRADE COMPLETA: {insumo_referencia['nome_area']} "
      f"({N_COMBINACOES} combinações, da melhor para a pior)")
print("=" * 104)
print(grade_completa.round(4).to_string(index=False))

In [ ]:
# ============================================================
# GRÁFICO: MAE ANTES (sem tuning) x AGORA (com tuning), por área
# Complementa a tabela "O tuning valeu a pena?" da etapa de tuning, usa a mesma
# `tuning`, sem recalcular nada.
# ============================================================
ordem_areas = tuning.sort_values("MAE_teste_padrao")["key_area"].tolist()
dados_grafico = tuning.set_index("key_area").loc[ordem_areas]

x = np.arange(len(ordem_areas))
largura = 0.35

fig, ax = plt.subplots(figsize=(11, 5.5))
barras_antes = ax.bar(x - largura / 2, dados_grafico["MAE_teste_padrao"], largura,
                       label="Antes (sem tuning)", color="#B0B0B0")
barras_agora = ax.bar(x + largura / 2, dados_grafico["MAE_teste_ajustado"], largura,
                       label="Agora (com tuning)", color="#4C72B0")

for barras in (barras_antes, barras_agora):
    ax.bar_label(barras, fmt="%.2f", padding=2, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([a.replace("_data.csv", "") for a in ordem_areas], rotation=20, ha="right")
ax.set_ylabel("MAE de teste (°C)")
ax.set_title("MAE antes x depois do tuning, por área", fontsize=12, fontweight="bold")
ax.legend()
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

pioraram = int((dados_grafico["MAE_teste_ajustado"] > dados_grafico["MAE_teste_padrao"]).sum())
print(f"Barra azul (agora) mais alta que a cinza (antes) em {pioraram} de {len(dados_grafico)} áreas"
      ": o tuning piorou o MAE de teste nessas.")

In [ ]:
# ============================================================
# GRÁFICO: R² ANTES (sem tuning) x AGORA (com tuning), por área
# Mesma dupla de barras da MAE acima, agora para R², usa a mesma `tuning`.
# ============================================================
dados_grafico_r2 = tuning.set_index("key_area").loc[ordem_areas]

x = np.arange(len(ordem_areas))
largura = 0.35

fig, ax = plt.subplots(figsize=(11, 5.5))
barras_antes_r2 = ax.bar(x - largura / 2, dados_grafico_r2["R2_teste_padrao"], largura,
                          label="Antes (sem tuning)", color="#B0B0B0")
barras_agora_r2 = ax.bar(x + largura / 2, dados_grafico_r2["R2_teste_ajustado"], largura,
                          label="Agora (com tuning)", color="#4C72B0")

for barras in (barras_antes_r2, barras_agora_r2):
    ax.bar_label(barras, fmt="%.3f", padding=2, fontsize=8)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([a.replace("_data.csv", "") for a in ordem_areas], rotation=20, ha="right")
ax.set_ylabel("R² de teste")
ax.set_title("R² antes x depois do tuning, por área", fontsize=12, fontweight="bold")
ax.legend()
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

pioraram_r2 = int((dados_grafico_r2["R2_teste_ajustado"] < dados_grafico_r2["R2_teste_padrao"]).sum())
print(f"R² caiu (ficou pior) em {pioraram_r2} de {len(dados_grafico_r2)} áreas com o tuning.")

## 11. Efeito de remover `board_position` e `material_coil_support`

A importância das variáveis calculada no treinamento já mostra, para a área
carregada no início do notebook, que `board_position` (0,011) e
`material_coil_support` (0,009) têm importância bem menor que as outras cinco
variáveis. Isso coloca a prova a hipótese **H2**, que
recomenda retirar variáveis redundantes do modelo.

Esta célula é esse teste, para o Random Forest e nas sete áreas: confirma que
a baixa importância se repete fora da área única, depois treina o mesmo modelo
(mesmos hiperparâmetros já escolhidos na etapa de tuning; não repete o Grid
Search, para isolar o efeito da remoção de variáveis do ruído do tuning já
discutido) com e sem as duas variáveis, e compara o MAE de teste.

In [ ]:
# ============================================================
# CÉLULA 11: EFEITO DE REMOVER board_position E material_coil_support
# Reusa os hiperparâmetros já escolhidos por área na etapa de tuning; não repete o
# Grid Search, para que a comparação isole o efeito das variáveis, e não do
# ruído do tuning já discutido.
# ============================================================
VARIAVEIS_BAIXA_IMPORTANCIA = ["board_position", "material_coil_support"]

print("=" * 100)
print("IMPORTÂNCIA DAS VARIÁVEIS COM O MODELO COMPLETO (7 variáveis), POR ÁREA")
print("=" * 100)
importancias_completo = pd.DataFrame({
    insumo["nome_area"]: pd.Series(insumo["modelo"].feature_importances_, index=VARIAVEIS_INDEPENDENTES)
    for insumo in INSUMOS_LOTE
}).T
print(importancias_completo.round(4).to_string())
print(f"\nMédia entre as sete áreas, da mais para a menos importante:\n"
      f"{importancias_completo.mean().sort_values(ascending=False).round(4).to_string()}")


def treinar_com_variaveis(insumo, variaveis):
    """Retreina com os MESMOS hiperparâmetros já escolhidos, só mudando as colunas de X."""
    params = insumo["busca"].best_params_
    modelo = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **params)
    modelo.fit(insumo["X_tr"][variaveis], insumo["Y_tr"])
    pred_teste = modelo.predict(insumo["X_te"][variaveis])
    mae = mean_absolute_error(insumo["Y_te"], pred_teste)
    r2 = r2_score(insumo["Y_te"], pred_teste)
    return modelo, mae, r2


VARIAVEIS_REDUZIDAS = [v for v in VARIAVEIS_INDEPENDENTES if v not in VARIAVEIS_BAIXA_IMPORTANCIA]

linhas, modelos_reduzidos = [], {}
for insumo in INSUMOS_LOTE:
    mae_completo = mean_absolute_error(insumo["Y_te"], insumo["previsoes_teste"])
    r2_completo = r2_score(insumo["Y_te"], insumo["previsoes_teste"])
    modelo_reduzido, mae_reduzido, r2_reduzido = treinar_com_variaveis(insumo, VARIAVEIS_REDUZIDAS)
    modelos_reduzidos[insumo["nome_area"]] = modelo_reduzido
    linhas.append({
        "key_area": insumo["nome_area"],
        "MAE_7_variaveis": mae_completo,
        "MAE_5_variaveis": mae_reduzido,
        "diferenca_C": mae_reduzido - mae_completo,
        "R2_7_variaveis": r2_completo,
        "R2_5_variaveis": r2_reduzido,
    })

comparativo_variaveis = pd.DataFrame(linhas)
print("\n" + "=" * 100)
print(f"REMOVER {VARIAVEIS_BAIXA_IMPORTANCIA}: MAE de teste, mesmos hiperparâmetros de cada área")
print("=" * 100)
print(comparativo_variaveis.round(3).to_string(index=False))
print(f"\nMelhorou (MAE menor) em {int((comparativo_variaveis['diferenca_C'] < 0).sum())} de "
      f"{len(comparativo_variaveis)} áreas, diferença média: "
      f"{comparativo_variaveis['diferenca_C'].mean():+.3f} °C")

print("\n" + "=" * 100)
print("IMPORTÂNCIA DAS VARIÁVEIS QUE SOBRARAM (5 variáveis), POR ÁREA")
print("=" * 100)
importancias_reduzido = pd.DataFrame({
    area: pd.Series(modelo.feature_importances_, index=VARIAVEIS_REDUZIDAS)
    for area, modelo in modelos_reduzidos.items()
}).T
print(importancias_reduzido.round(4).to_string())
print(f"\nMédia entre as sete áreas, da mais para a menos importante:\n"
      f"{importancias_reduzido.mean().sort_values(ascending=False).round(4).to_string()}")

In [ ]:
# ============================================================
# GRÁFICO: MAE COM 7 VARIÁVEIS x SEM board_position/material_coil_support
# Complementa a tabela da seção anterior, usa `comparativo_variaveis`, sem
# recalcular nada.
# ============================================================
ordem_areas_var = comparativo_variaveis.sort_values("MAE_7_variaveis")["key_area"].tolist()
dados_grafico_var = comparativo_variaveis.set_index("key_area").loc[ordem_areas_var]

x = np.arange(len(ordem_areas_var))
largura = 0.35

fig, ax = plt.subplots(figsize=(11, 5.5))
barras_sem_remover = ax.bar(x - largura / 2, dados_grafico_var["MAE_7_variaveis"], largura,
                             label="Sem remover (7 variáveis)", color="#B0B0B0")
barras_depois = ax.bar(x + largura / 2, dados_grafico_var["MAE_5_variaveis"], largura,
                        label="Depois de remover (5 variáveis)", color="#4C72B0")

for barras in (barras_sem_remover, barras_depois):
    ax.bar_label(barras, fmt="%.2f", padding=2, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([a.replace("_data.csv", "") for a in ordem_areas_var], rotation=20, ha="right")
ax.set_ylabel("MAE de teste (°C)")
ax.set_title(
    "MAE sem remover x depois de remover board_position/material_coil_support, por área\n"
    "(quanto menor, melhor ↓)",
    fontsize=12, fontweight="bold")
ax.legend()
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

melhoraram = int((dados_grafico_var["MAE_5_variaveis"] < dados_grafico_var["MAE_7_variaveis"]).sum())
print(f"Barra azul (depois de remover) mais baixa que a cinza (sem remover) em "
      f"{melhoraram} de {len(dados_grafico_var)} áreas: a remoção ajudou nessas.")

In [ ]:
# ============================================================
# GRÁFICO: R² COM 7 VARIÁVEIS x SEM board_position/material_coil_support
# Mesma dupla de barras da MAE acima, agora para R², usa a mesma
# `comparativo_variaveis`.
# ============================================================
dados_grafico_var_r2 = comparativo_variaveis.set_index("key_area").loc[ordem_areas_var]

x = np.arange(len(ordem_areas_var))
largura = 0.35

fig, ax = plt.subplots(figsize=(11, 5.5))
barras_sem_remover_r2 = ax.bar(x - largura / 2, dados_grafico_var_r2["R2_7_variaveis"], largura,
                                label="Sem remover (7 variáveis)", color="#B0B0B0")
barras_depois_r2 = ax.bar(x + largura / 2, dados_grafico_var_r2["R2_5_variaveis"], largura,
                           label="Depois de remover (5 variáveis)", color="#4C72B0")

for barras in (barras_sem_remover_r2, barras_depois_r2):
    ax.bar_label(barras, fmt="%.3f", padding=2, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([a.replace("_data.csv", "") for a in ordem_areas_var], rotation=20, ha="right")
ax.set_ylabel("R² de teste")
ax.set_title(
    "R² sem remover x depois de remover board_position/material_coil_support, por área\n"
    "(quanto maior, melhor ↑)",
    fontsize=12, fontweight="bold")
ax.legend()
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

melhoraram_var_r2 = int((dados_grafico_var_r2["R2_5_variaveis"] > dados_grafico_var_r2["R2_7_variaveis"]).sum())
print(f"R² melhorou (subiu) em {melhoraram_var_r2} de {len(dados_grafico_var_r2)} áreas ao remover as variáveis.")

## 12. Por que o R² e o MAE se comportam assim?

As seções anteriores (sobreajuste em lote, tuning de hiperparâmetros e remoção
de variáveis) mostraram tuning ordenando ruído e R² instável/negativo em várias
áreas. Antes de tentar mais uma correção, vale confirmar: isso é um problema do
nosso pipeline, ou o comportamento esperado de Random Forest com poucos dados?
Uma pesquisa externa confirma a segunda hipótese, com
respaldo em três pontos.

**1. R² é enviesado, e o viés cresce quando a amostra é pequena.** É
documentado que R² ajustado existe justamente para corrigir esse viés, e que
o R² **pode ficar zero ou negativo quando um modelo com variáveis pouco
informativas é ajustado a uma amostra pequena demais**: é exatamente o que
vimos em `ipc_comp` (16 linhas de treino, R² a −0,81).

**2. Random Forest precisa de um número mínimo de árvores/dados para a
variância da sua própria estimativa se estabilizar.** A pesquisa mostra que
a estimativa de variância interna do modelo fica excessivamente grande com
poucas árvores, e que são necessárias da ordem de `n` (tamanho
da amostra) árvores para uma estimativa estável.

**3. Com poucas variáveis, o Random Forest perde seu próprio mecanismo
diferencial.** `max_features` sorteia um subconjunto de variáveis a cada
divisão para descorrelacionar as árvores: é o que separa Random Forest de um
bagging comum. Com 1 variável, não sobra o que sortear: o modelo degenera
para bagging de árvores idênticas. Com nossas 7 variáveis e
`max_features="sqrt"` (≈2-3 sorteadas por vez), esse mecanismo já opera perto
do limite inferior, o que explica por que a "aleatoriedade útil" do algoritmo
rende pouco aqui.

**O caso externo mais direto:** o exemplo clássico de Random Forest com uma
única variável numérica (nível de cargo → salário, dataset `Position_Salaries`,
usado em milhares de tutoriais) tem **10 linhas**. Nele, o R² sobre o próprio
ajuste é 0,9878, mas o R² OOB (*out-of-bag*, a estimativa honesta de
generalização que o próprio Random Forest calcula internamente) é **0,2281**,
quase 0,76 pontos de diferença, com apenas 10 linhas. É o mesmo padrão que
vimos aqui: métrica de treino ótima, métrica de generalização muito pior.

**Conclusão da parte 1:** o comportamento documentado neste notebook não é
peculiar ao nosso pipeline: é o esperado quando a razão variáveis/amostras é
desfavorável, replicável até em exemplos didáticos de 10 linhas com uma única
variável. O teto é estrutural (volume de dados); nenhum ajuste de
hiperparâmetro ou seleção de variável resolve isso; só mais ensaios
físicos/CFD.


### O que os valores em si dizem sobre o nosso caso

Entender por que a métrica oscila não responde a outra pergunta: os
valores que de fato obtivemos são bons ou ruins para o que o parceiro
precisa?

**MAE: a régua já existe, a margem de 10% combinada com o parceiro.** Um erro
de "2 a 10 °C" não diz nada sozinho; o que importa é quão grande esse erro é
perto da faixa de temperatura de cada sensor. É exatamente o que o MAPE e o
`dentro_de_10%` já medem. Por essa régua:

* `oven_hmi`           (MAPE 3,3%, 100% das previsões dentro de 10%)

* `main_vent_sys`      (5,0%, 80%)

* `ct_knobs`           (5,1%, 83%)

* `ipc_cooling`        (6,0%, 83%)  

* `mainback`           (9,3%, 64% dentro de 10%) 

* `ipc_comp`           (11,6%, 54% dentro de 10%)

**R²: a régua é "melhor que citar a média histórica da área", não um valor
absoluto de 0 a 1.** Um R² de 0,3 a 0,6 parece modesto pelos padrões usuais de regressão, mas para o
uso real do modelo, sinalizar o patamar térmico esperado e priorizar quais
ensaios físicos rodar primeiro, a "triagem térmica" já delimitada na
documentação, esse nível de R² já é suficiente: o modelo distingue
configurações com folga térmica real de configurações mais próximas do
`active_limit`, mesmo sem acertar o valor exato. O alerta real está nas áreas
com R² negativo, ali o modelo é **pior** do que simplesmente citar a temperatura média histórica da
área. Dessa forma o modelo ainda não serve para ranqueamento fino.


## 14. Quantidade de árvores x erro: as sete áreas de uma vez

`n_estimators` é o hiperparâmetro que a busca em grade mais escolheu alto (300
a 800, na etapa de tuning). Vale conferir se esse valor alto é ganho real ou
apenas a busca escolhendo dentro do ruído: esta célula varia **só** o número de
árvores, mantendo fixos os demais hiperparâmetros de cada área, e mede o MAE
de teste em cada ponto.

São dois painéis lado a lado porque as áreas operam em patamares de erro muito
diferentes (de ~2 °C a ~10 °C):

- **Esquerda, MAE absoluto (°C):** a magnitude real do erro de cada área.
- **Direita, MAE relativo ao melhor ponto da própria área (%):** a *forma* da
  curva, comparável entre áreas. É aqui que se vê a partir de quantas árvores
  a curva estabiliza, e, portanto, quanto do `n_estimators` alto escolhido
  pela busca é ganho e quanto é excesso.

In [ ]:
# ============================================================
# CÉLULA 14: CURVA DE QUANTIDADE DE ÁRVORES x MAE, NAS SETE ÁREAS
# Varia apenas n_estimators; os demais hiperparâmetros são os já escolhidos
# por área na etapa de tuning, para isolar o efeito do número de árvores.
# ============================================================
VALORES_ARVORES = [10, 25, 50, 100, 200, 300, 500, 800]

# Paleta categórica de 7 cores, em ordem fixa (uma por área).
CORES_AREAS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]

linhas_curva = []
for insumo in INSUMOS_LOTE:
    params_area = dict(insumo["busca"].best_params_)
    for n_arvores in VALORES_ARVORES:
        params_area["n_estimators"] = n_arvores
        modelo_n = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **params_area)
        modelo_n.fit(insumo["X_tr"], insumo["Y_tr"])
        linhas_curva.append({
            "key_area": insumo["nome_area"],
            "n_estimators": n_arvores,
            "MAE_teste": mean_absolute_error(insumo["Y_te"], modelo_n.predict(insumo["X_te"])),
        })

curva_arvores = pd.DataFrame(linhas_curva)
tabela_curva = curva_arvores.pivot(index="n_estimators", columns="key_area", values="MAE_teste")
tabela_curva.columns = [c.replace("_data.csv", "") for c in tabela_curva.columns]

print("=" * 112)
print("MAE DE TESTE (°C) POR QUANTIDADE DE ÁRVORES: TODAS AS ÁREAS")
print("=" * 112)
print(tabela_curva.round(3).to_string())

melhor_n = tabela_curva.idxmin()
print("\nQuantidade de árvores que minimiza o MAE em cada área:")
for area, n in melhor_n.items():
    ganho = tabela_curva[area].max() - tabela_curva[area].min()
    print(f"   {area:16s} melhor n = {n:4d}  |  amplitude da curva inteira: {ganho:.3f} °C")
print(f"\nAmplitude média entre o melhor e o pior número de árvores: "
      f"{(tabela_curva.max() - tabela_curva.min()).mean():.3f} °C")

fig, (ax_abs, ax_rel) = plt.subplots(1, 2, figsize=(13, 5.5))

for i, area in enumerate(tabela_curva.columns):
    cor = CORES_AREAS[i % len(CORES_AREAS)]
    ax_abs.plot(tabela_curva.index, tabela_curva[area], marker="o", markersize=4,
                linewidth=2, color=cor, label=area)
    relativo = (tabela_curva[area] / tabela_curva[area].min() - 1) * 100
    ax_rel.plot(tabela_curva.index, relativo, marker="o", markersize=4,
                linewidth=2, color=cor, label=area)

ax_abs.set_xlabel("Quantidade de árvores (n_estimators)")
ax_abs.set_ylabel("MAE de teste (°C)")
ax_abs.set_title("MAE absoluto\n(quanto menor, melhor ↓)", fontsize=11, fontweight="bold")

ax_rel.set_xlabel("Quantidade de árvores (n_estimators)")
ax_rel.set_ylabel("MAE acima do melhor ponto da própria área (%)")
ax_rel.set_title("MAE relativo ao melhor ponto de cada área\n(quanto menor, melhor ↓)",
                 fontsize=11, fontweight="bold")

for eixo in (ax_abs, ax_rel):
    eixo.set_xscale("log")
    eixo.set_xticks(VALORES_ARVORES)
    eixo.set_xticklabels(VALORES_ARVORES)
    for lado in ("top", "right"):
        eixo.spines[lado].set_visible(False)

ax_rel.legend(title="key_area", fontsize=8)
plt.tight_layout()
plt.show()

## 15. Temperatura prevista x temperatura medida

O gráfico de dispersão previsto-contra-medido é o diagnóstico mais direto que
existe para regressão: cada ponto é um termopar num ensaio de teste, o eixo
horizontal traz a temperatura medida e o vertical a prevista pelo modelo.

Como ler:

- **Pontos sobre a diagonal** (linha `y = x`) são previsões perfeitas.
- **Ponto acima da diagonal** = o modelo previu
  mais quente do que foi medido a temperatura. É o erro "seguro" para o parceiro: manda
  checar uma configuração que na prática estava mais fria.
- **Ponto abaixo da diagonal** = o modelo **subestimou**. É o erro perigoso,
  porque pode liberar como segura uma configuração que na realidade está mais
  perto do `active_limit`.
- **Dispersão vertical** em torno da diagonal = magnitude do erro; **desvio
  sistemático** da nuvem para um dos lados = viés do modelo naquela faixa.

Os pontos são coloridos por `key_area` porque cada região opera num patamar
térmico diferente: o gráfico mostra também se o modelo acerta melhor nas
faixas frias ou nas quentes.

In [ ]:
# ============================================================
# CÉLULA 15: TEMPERATURA PREVISTA x TEMPERATURA MEDIDA
# Usa as previsões de teste já calculadas na execução em lote.
# ============================================================
pontos = []
for insumo in INSUMOS_LOTE:
    medido = insumo["Y_te"].values.ravel()
    previsto = np.asarray(insumo["previsoes_teste"]).ravel()
    valido = ~np.isnan(medido) & ~np.isnan(previsto)
    pontos.append(pd.DataFrame({
        "key_area": insumo["nome_area"].replace("_data.csv", ""),
        "medido": medido[valido],
        "previsto": previsto[valido],
    }))

previsto_vs_medido = pd.concat(pontos, ignore_index=True)
previsto_vs_medido["erro"] = previsto_vs_medido["previsto"] - previsto_vs_medido["medido"]

limite_min = min(previsto_vs_medido["medido"].min(), previsto_vs_medido["previsto"].min()) - 5
limite_max = max(previsto_vs_medido["medido"].max(), previsto_vs_medido["previsto"].max()) + 5

fig, ax = plt.subplots(figsize=(9, 8))

eixo_diagonal = np.array([limite_min, limite_max])
ax.plot(eixo_diagonal, eixo_diagonal,
        color="#595959", linewidth=1.2, zorder=1)
ax.annotate("previsão perfeita (y = x)", xy=(limite_max, limite_max),
            xytext=(-8, -16), textcoords="offset points",
            ha="right", fontsize=8, color="#595959")

# Faixa de ±10% (margem combinada com o parceiro, Seção 8): como é percentual
# do valor medido, a faixa se abre conforme a temperatura aumenta, em vez de
# manter largura constante.
eixo_positivo = eixo_diagonal[eixo_diagonal > 0]
if len(eixo_positivo) == 2:
    ax.fill_between(eixo_positivo, eixo_positivo * 0.90, eixo_positivo * 1.10,
                     color="#4C72B0", alpha=0.12, zorder=0,
                     label="margem de ±10% (medido)")

for i, (area, grupo) in enumerate(previsto_vs_medido.groupby("key_area", sort=False)):
    ax.scatter(grupo["medido"], grupo["previsto"], s=26, alpha=0.75,
               color=CORES_AREAS[i % len(CORES_AREAS)], label=area,
               edgecolors="white", linewidths=0.5, zorder=2)

ax.text(0.03, 0.97, "acima da diagonal:\nsuperestimou (erro seguro)",
        transform=ax.transAxes, fontsize=8, color="#595959", va="top")
ax.text(0.97, 0.04, "abaixo da diagonal:\nsubestimou (erro perigoso)",
        transform=ax.transAxes, fontsize=8, color="#C44E52", ha="right", va="bottom")

ax.set_xlim(limite_min, limite_max)
ax.set_ylim(limite_min, limite_max)
ax.set_aspect("equal")
ax.set_xlabel("Temperatura medida (°C)")
ax.set_ylabel("Temperatura prevista (°C)")
ax.set_title("Temperatura prevista x medida, conjunto de teste, todas as áreas",
             fontsize=12, fontweight="bold")
# Legenda no canto superior esquerdo, logo abaixo da anotação, para não cobrir
# a região abaixo da diagonal (onde ficam as subestimativas).
ax.legend(title="key_area", fontsize=8, loc="upper left", bbox_to_anchor=(0.0, 0.88))
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()

subestimadas = (previsto_vs_medido["erro"] < 0).mean() * 100
print(f"Previsões abaixo da medição (subestimou, lado perigoso): {subestimadas:.1f}%")
print(f"Previsões acima da medição (superestimou, lado seguro):  {100 - subestimadas:.1f}%")
print(f"Viés médio (previsto - medido): {previsto_vs_medido['erro'].mean():+.2f} °C")

print("\nErro por área (previsto - medido):")
print(previsto_vs_medido.groupby("key_area")["erro"]
      .agg(vies_medio_C="mean", erro_absoluto_medio_C=lambda s: s.abs().mean(),
           pior_subestimativa_C="min", pior_superestimativa_C="max")
      .sort_values("erro_absoluto_medio_C").round(2).to_string())

## 16. Fechando a distância até o piso: alinhar o critério do modelo à métrica

Mantendo apenas as 7 variáveis, sem recuperar nada do `reason` e sem novos
ensaios, ainda há uma distância até o piso de erro irredutível calculado antes.
Boa parte dela vem de uma incoerência que passou despercebida em todas as
buscas anteriores:

**o modelo é treinado para minimizar erro quadrático, mas é avaliado por MAE.**

O `RandomForestRegressor` usa `criterion="squared_error"` dividindo os
nós minimizando variância e **devolve a média** das amostras de cada folha. Só
que o previsor constante que minimiza o **erro absoluto** dentro de um grupo
não é a média, é a **mediana**. Esse piso é justamente o desvio em torno da
mediana; enquanto o modelo entregar médias, ele persegue um alvo diferente do
que a métrica cobra.

O sklearn tem `criterion="absolute_error"`: as árvores passam a dividir
minimizando MAE e as folhas passam a devolver a mediana. Esse parâmetro
nunca esteve na grade de busca, então
nenhuma das 672 combinações testadas na etapa de tuning chegou perto dele.

Esta célula compara três abordagens, todas com as mesmas 7 variáveis e a mesma
divisão treino/teste:

- **A (atual):** o vencedor da busca em grade (critério quadrático).
- **B (critério alinhado):** `criterion="absolute_error"`, demais parâmetros no
  padrão. Uma linha de mudança.
- **C (rumo ao piso):** critério MAE + `bootstrap=False` + sem limite de
  profundidade, forçando o modelo a isolar cada configuração. É a configuração
  que mais se aproxima do piso nas configurações já vistas, e a que mais
  erra nas configurações novas, que é o caso de uso real do parceiro.

In [ ]:
# ============================================================
# CÉLULA 17 — ALINHAR O CRITÉRIO DO MODELO À MÉTRICA AVALIADA
# Só as 7 variáveis, mesma divisão treino/teste. A única mudança é o critério
# de divisão das árvores (e, na variante C, a regularização).
# ============================================================
VARIANTES_CRITERIO = {
    "B_criterio_MAE": dict(n_estimators=300, criterion="absolute_error"),
    "C_regularizado": dict(n_estimators=300, criterion="absolute_error",
                           max_depth=None, min_samples_leaf=1,
                           max_features=None, bootstrap=False),
}

linhas_criterio = []
for insumo in INSUMOS_LOTE:
    registro = {
        "key_area": insumo["nome_area"],
        "A_atual": mean_absolute_error(insumo["Y_te"], insumo["previsoes_teste"]),
    }

    for nome_variante, params in VARIANTES_CRITERIO.items():
        modelo_v = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **params)
        modelo_v.fit(insumo["X_tr"], insumo["Y_tr"])
        pred_v = modelo_v.predict(insumo["X_te"])
        registro[nome_variante] = mean_absolute_error(insumo["Y_te"], pred_v)

    linhas_criterio.append(registro)

comparativo_criterio = pd.DataFrame(linhas_criterio)
comparativo_criterio["ganho_B_C"] = (comparativo_criterio["A_atual"]
                                     - comparativo_criterio["B_criterio_MAE"])

print("=" * 112)
print("MAE DE TESTE: MODELO ATUAL x CRITÉRIO ALINHADO AO MAE x VARIANTE REGULARIZADA")
print("=" * 112)
print(comparativo_criterio[["key_area", "A_atual", "B_criterio_MAE", "C_regularizado",
                            "ganho_B_C"]]
      .round(3).to_string(index=False))

ganhos = comparativo_criterio["ganho_B_C"]
print(f"\nB (uma linha: criterion='absolute_error') melhorou em {int((ganhos > 0).sum())} de "
      f"{len(ganhos)} áreas — ganho médio {ganhos.mean():+.3f} °C")

# Gráfico: barras atual x critério MAE, por área.
ordem_crit = comparativo_criterio.sort_values("A_atual")["key_area"].tolist()
dados_crit = comparativo_criterio.set_index("key_area").loc[ordem_crit]
x = np.arange(len(ordem_crit))
largura = 0.35

fig, ax = plt.subplots(figsize=(11.5, 5.5))
barras_a = ax.bar(x - largura / 2, dados_crit["A_atual"], largura,
                  label="A — atual (critério quadrático)", color="#B0B0B0")
barras_b = ax.bar(x + largura / 2, dados_crit["B_criterio_MAE"], largura,
                  label="B — critério alinhado ao MAE (absolute_error)", color="#4C72B0")
for barras in (barras_a, barras_b):
    ax.bar_label(barras, fmt="%.2f", padding=2, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([a.replace("_data.csv", "") for a in ordem_crit], rotation=20, ha="right")
ax.set_ylabel("MAE de teste (°C)")
ax.set_title(
    "Alinhar o critério do modelo à métrica avaliada, por área\n"
    "(quanto menor, melhor ↓)",
    fontsize=12, fontweight="bold")
ax.legend(fontsize=8)
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# CÉLULA 18: MÉDIA x MEDIANA, MAE E PREVISTO x MEDIDO, LADO A LADO
# Reusa `comparativo_criterio` da comparação de critérios anterior para o MAE;
# recalcula o modelo de mediana só para guardar as previsões ponto-a-ponto (a
# comparação anterior guardou apenas o MAE agregado).
# ============================================================
PARAMS_MEDIANA = dict(n_estimators=300, criterion="absolute_error")

pontos_mediana = []
for insumo in INSUMOS_LOTE:
    modelo_mediana = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **PARAMS_MEDIANA)
    modelo_mediana.fit(insumo["X_tr"], insumo["Y_tr"])
    pred = modelo_mediana.predict(insumo["X_te"])

    medido = insumo["Y_te"].values.ravel()
    previsto = np.asarray(pred).ravel()
    valido = ~np.isnan(medido) & ~np.isnan(previsto)
    pontos_mediana.append(pd.DataFrame({
        "key_area": insumo["nome_area"].replace("_data.csv", ""),
        "medido": medido[valido],
        "previsto": previsto[valido],
    }))

previsto_vs_medido_mediana = pd.concat(pontos_mediana, ignore_index=True)
previsto_vs_medido_mediana["erro"] = (previsto_vs_medido_mediana["previsto"]
                                      - previsto_vs_medido_mediana["medido"])

fig, (ax_mae, ax_disp) = plt.subplots(1, 2, figsize=(15, 6.5))

# --- painel esquerdo: MAE média x mediana, por área --------------------
ordem_media_mediana = comparativo_criterio.sort_values("A_atual")["key_area"].tolist()
dados_mm = comparativo_criterio.set_index("key_area").loc[ordem_media_mediana]
x = np.arange(len(ordem_media_mediana))
largura = 0.35

barras_media = ax_mae.bar(x - largura / 2, dados_mm["A_atual"], largura,
                          label="Critério: média (squared_error)", color="#B0B0B0")
barras_mediana = ax_mae.bar(x + largura / 2, dados_mm["B_criterio_MAE"], largura,
                            label="Critério: mediana (absolute_error)", color="#4C72B0")
for barras in (barras_media, barras_mediana):
    ax_mae.bar_label(barras, fmt="%.2f", padding=2, fontsize=7)

ax_mae.set_xticks(x)
ax_mae.set_xticklabels([a.replace("_data.csv", "") for a in ordem_media_mediana],
                       rotation=20, ha="right")
ax_mae.set_ylabel("MAE de teste (°C)")
ax_mae.set_title("MAE: média x mediana\n(quanto menor, melhor ↓)",
                 fontsize=11, fontweight="bold")
ax_mae.legend(fontsize=8)
for lado in ("top", "right"):
    ax_mae.spines[lado].set_visible(False)

# --- painel direito: previsto x medido, modelo de mediana ---------------
limite_min = min(previsto_vs_medido_mediana["medido"].min(),
                 previsto_vs_medido_mediana["previsto"].min()) - 5
limite_max = max(previsto_vs_medido_mediana["medido"].max(),
                 previsto_vs_medido_mediana["previsto"].max()) + 5

ax_disp.plot([limite_min, limite_max], [limite_min, limite_max],
            color="#595959", linewidth=1.2, zorder=1)
ax_disp.annotate("previsão perfeita (y = x)", xy=(limite_max, limite_max),
                 xytext=(-8, -14), textcoords="offset points",
                 ha="right", fontsize=8, color="#595959")

for i, (area, grupo) in enumerate(previsto_vs_medido_mediana.groupby("key_area", sort=False)):
    ax_disp.scatter(grupo["medido"], grupo["previsto"], s=24, alpha=0.75,
                    color=CORES_AREAS[i % len(CORES_AREAS)], label=area,
                    edgecolors="white", linewidths=0.5, zorder=2)

ax_disp.set_xlim(limite_min, limite_max)
ax_disp.set_ylim(limite_min, limite_max)
ax_disp.set_aspect("equal")
ax_disp.set_xlabel("Temperatura medida (°C)")
ax_disp.set_ylabel("Temperatura prevista (°C)")
ax_disp.set_title("Previsto x medido, critério de mediana\n(comparar com o gráfico anterior)",
                  fontsize=11, fontweight="bold")
ax_disp.legend(title="key_area", fontsize=7, loc="upper left", bbox_to_anchor=(0.0, 0.85))
for lado in ("top", "right"):
    ax_disp.spines[lado].set_visible(False)

plt.tight_layout()
plt.show()

# --- comparação numérica com o gráfico anterior (critério de média) ------------
subestimadas_mediana = (previsto_vs_medido_mediana["erro"] < 0).mean() * 100
vies_mediana = previsto_vs_medido_mediana["erro"].mean()

print("=" * 100)
print("COMPARAÇÃO COM A CÉLULA 15 (critério de média)")
print("=" * 100)
print(f"{'':22s}{'média (critério padrão)':>20s}{'mediana (aqui)':>20s}")
print(f"{'Viés médio (°C)':22s}{previsto_vs_medido['erro'].mean():>20.2f}"
      f"{vies_mediana:>20.2f}")
print(f"{'% subestimado':22s}{subestimadas:>19.1f}%{subestimadas_mediana:>19.1f}%")
print(f"{'Pior subestimativa':22s}{previsto_vs_medido['erro'].min():>20.2f}"
      f"{previsto_vs_medido_mediana['erro'].min():>20.2f}")
print(f"{'Pior superestimativa':22s}{previsto_vs_medido['erro'].max():>20.2f}"
      f"{previsto_vs_medido_mediana['erro'].max():>20.2f}")

print("\nErro por área (previsto - medido), critério de mediana:")
print(previsto_vs_medido_mediana.groupby("key_area")["erro"]
      .agg(vies_medio_C="mean", erro_absoluto_medio_C=lambda s: s.abs().mean(),
           pior_subestimativa_C="min", pior_superestimativa_C="max")
      .sort_values("erro_absoluto_medio_C").round(2).to_string())

- **Painel esquerdo:** MAE de teste, mesmo modelo e mesma divisão
  treino/teste, variando só o critério das árvores.
- **Painel direito:** previsto x medido usando o modelo de **mediana**, no
  mesmo formato do gráfico anterior, para comparar diretamente os dois.

## 18. Estimador híbrido: mediana nas caudas, média no meio

- Se a **média da folha** cai **dentro** do intervalo [Q1, Q3] usa-se a
  **média** (mais eficiente estatisticamente quando os dados são bem
  comportados).
- Se a média cai **fora** do [Q1, Q3], sinal de que algum valor extremo da
  própria folha está puxando a média para longe do
  centro da distribuição, usa-se a **mediana**.

Isso é o que uma *Quantile Regression Forest* (Meinshausen, 2006) faz de
forma mais geral: guarda a distribuição empírica de valores de treino em cada
folha e agrega por estatística de ordem, em vez de só a média. Aqui a regra é
mais simples, uma escolha binária entre média e mediana, mas o mecanismo de
base é o
mesmo.

In [ ]:
# ============================================================
# CÉLULA 20: ESTIMADOR HÍBRIDO (MEDIANA FORA DO IQR, MÉDIA DENTRO)
# ============================================================
def estatisticas_por_folha(arvore, X_treino_arr, Y_treino_arr):
    """Para cada folha da árvore, decide entre média e mediana pela regra do IQR."""
    ids_folha = arvore.tree_.apply(X_treino_arr)
    estatisticas = {}
    for folha in np.unique(ids_folha):
        valores = Y_treino_arr[ids_folha == folha]          # (n_na_folha, n_alvos)
        media = valores.mean(axis=0)
        mediana = np.median(valores, axis=0)
        q1 = np.percentile(valores, 25, axis=0)
        q3 = np.percentile(valores, 75, axis=0)
        fora_do_iqr = (media < q1) | (media > q3)
        estatisticas[folha] = np.where(fora_do_iqr, mediana, media)
    return estatisticas


def prever_hibrido(modelo_rf, X_treino, Y_treino, X_novo):
    """Média das previsões de cada árvore, cada uma usando a regra híbrida por folha."""
    X_treino_arr = X_treino.values.astype(np.float32)
    X_novo_arr = X_novo.values.astype(np.float32)
    Y_treino_arr = Y_treino.values.astype(float)

    previsoes_por_arvore = []
    for arvore in modelo_rf.estimators_:
        estatisticas = estatisticas_por_folha(arvore, X_treino_arr, Y_treino_arr)
        ids_folha_novo = arvore.tree_.apply(X_novo_arr)
        previsoes_por_arvore.append(np.array([estatisticas[f] for f in ids_folha_novo]))

    return np.mean(previsoes_por_arvore, axis=0)


N_ARVORES_HIBRIDO = 300

linhas_hibrido = []
pontos_hibrido = []
for insumo in INSUMOS_LOTE:
    modelo_base = RandomForestRegressor(n_estimators=N_ARVORES_HIBRIDO,
                                         random_state=SEMENTE, n_jobs=-1)
    modelo_base.fit(insumo["X_tr"], insumo["Y_tr"])

    pred_hibrida = prever_hibrido(modelo_base, insumo["X_tr"], insumo["Y_tr"], insumo["X_te"])
    mae_hibrido = mean_absolute_error(insumo["Y_te"], pred_hibrida)

    linhas_hibrido.append({"key_area": insumo["nome_area"], "C_hibrido": mae_hibrido})

    medido = insumo["Y_te"].values.ravel()
    previsto = pred_hibrida.ravel()
    valido = ~np.isnan(medido) & ~np.isnan(previsto)
    pontos_hibrido.append(pd.DataFrame({
        "key_area": insumo["nome_area"].replace("_data.csv", ""),
        "medido": medido[valido],
        "previsto": previsto[valido],
    }))

resultado_hibrido = pd.DataFrame(linhas_hibrido)
previsto_vs_medido_hibrido = pd.concat(pontos_hibrido, ignore_index=True)
previsto_vs_medido_hibrido["erro"] = (previsto_vs_medido_hibrido["previsto"]
                                      - previsto_vs_medido_hibrido["medido"])

# Junta com média e mediana calculadas na comparação de critérios anterior, para
# a comparação de três vias.
tres_vias = comparativo_criterio[["key_area", "A_atual", "B_criterio_MAE"]].merge(
    resultado_hibrido, on="key_area"
).rename(columns={"A_atual": "média", "B_criterio_MAE": "mediana", "C_hibrido": "híbrido"})

print("=" * 100)
print("MAE DE TESTE: MÉDIA x MEDIANA x HÍBRIDO (mediana fora do IQR da folha, média dentro)")
print("=" * 100)
print(tres_vias.round(3).to_string(index=False))
print(f"\nHíbrido melhor que a média em {int((tres_vias['híbrido'] < tres_vias['média']).sum())} de "
      f"{len(tres_vias)} áreas")
print(f"Híbrido melhor que a mediana em {int((tres_vias['híbrido'] < tres_vias['mediana']).sum())} de "
      f"{len(tres_vias)} áreas")
print(f"Híbrido é o MELHOR dos três em {int((tres_vias[['média','mediana','híbrido']].idxmin(axis=1) == 'híbrido').sum())} "
      f"de {len(tres_vias)} áreas")

# ---- gráfico: MAE (três barras) + previsto x medido do híbrido ----
fig, (ax_mae, ax_disp) = plt.subplots(1, 2, figsize=(16, 6.5))

ordem_tres = tres_vias.sort_values("média")["key_area"].tolist()
dados_tres = tres_vias.set_index("key_area").loc[ordem_tres]
x = np.arange(len(ordem_tres))
largura = 0.27

cores_criterio = {"média": "#B0B0B0", "mediana": "#4C72B0", "híbrido": "#1baf7a"}
for i, criterio in enumerate(["média", "mediana", "híbrido"]):
    offset = (i - 1) * largura
    barras = ax_mae.bar(x + offset, dados_tres[criterio], largura,
                        label=criterio, color=cores_criterio[criterio])
    ax_mae.bar_label(barras, fmt="%.2f", padding=2, fontsize=6.5)

ax_mae.set_xticks(x)
ax_mae.set_xticklabels([a.replace("_data.csv", "") for a in ordem_tres], rotation=20, ha="right")
ax_mae.set_ylabel("MAE de teste (°C)")
ax_mae.set_title("MAE: média x mediana x híbrido\n(quanto menor, melhor ↓)",
                 fontsize=11, fontweight="bold")
ax_mae.legend(fontsize=8)
for lado in ("top", "right"):
    ax_mae.spines[lado].set_visible(False)

limite_min = min(previsto_vs_medido_hibrido["medido"].min(),
                 previsto_vs_medido_hibrido["previsto"].min()) - 5
limite_max = max(previsto_vs_medido_hibrido["medido"].max(),
                 previsto_vs_medido_hibrido["previsto"].max()) + 5

ax_disp.plot([limite_min, limite_max], [limite_min, limite_max],
            color="#595959", linewidth=1.2, zorder=1)
ax_disp.annotate("previsão perfeita (y = x)", xy=(limite_max, limite_max),
                 xytext=(-8, -14), textcoords="offset points",
                 ha="right", fontsize=8, color="#595959")

for i, (area, grupo) in enumerate(previsto_vs_medido_hibrido.groupby("key_area", sort=False)):
    ax_disp.scatter(grupo["medido"], grupo["previsto"], s=24, alpha=0.75,
                    color=CORES_AREAS[i % len(CORES_AREAS)], label=area,
                    edgecolors="white", linewidths=0.5, zorder=2)

ax_disp.set_xlim(limite_min, limite_max)
ax_disp.set_ylim(limite_min, limite_max)
ax_disp.set_aspect("equal")
ax_disp.set_xlabel("Temperatura medida (°C)")
ax_disp.set_ylabel("Temperatura prevista (°C)")
ax_disp.set_title("Previsto x medido, estimador híbrido\n(comparar com os gráficos anteriores)",
                  fontsize=11, fontweight="bold")
ax_disp.legend(title="key_area", fontsize=7, loc="upper left", bbox_to_anchor=(0.0, 0.85))
for lado in ("top", "right"):
    ax_disp.spines[lado].set_visible(False)

plt.tight_layout()
plt.show()

print("\nErro por área (previsto - medido), estimador híbrido:")
print(previsto_vs_medido_hibrido.groupby("key_area")["erro"]
      .agg(vies_medio_C="mean", erro_absoluto_medio_C=lambda s: s.abs().mean(),
           pior_subestimativa_C="min", pior_superestimativa_C="max")
      .sort_values("erro_absoluto_medio_C").round(2).to_string())

## Célula 21: Validação Leave-One-Group-Out (por configuração) vs. holdout fixo

Com apenas 32–43 linhas por área após o tratamento, o `GridSearchCV` já usa só 2–3 folds (como definido no
treinamento e reaproveitado na execução em lote) e a avaliação final depende de
um único corte fixo de 20% para teste (definido na separação
treino/teste/verificação), pouco robusto com tão poucos dados.

Esta célula reavalia os modelos já ajustados com **Leave-One-Group-Out
(LOGO)**. Vale registrar como o grupo é de fato montado no código: cada linha é tratada como seu próprio
grupo por construção, e não porque cada linha seja uma configuração distinta
É uma simplificação assumida aqui: o
agrupamento não junta as réplicas de uma mesma configuração, então o LOGO
desta célula se comporta na prática como um Leave-One-Out sobre as linhas de
cada área. O ganho em relação ao holdout continua valendo: usa **todas** as
linhas como teste (uma de cada vez), em vez de depender de um único corte fixo
de 20%.

Os hiperparâmetros usados em cada fold são os já escolhidos pelo Grid Search
feito na execução em lote: o objetivo é
comparar a **estratégia de validação**, não repetir o tuning.

In [ ]:
# ============================================================
# CÉLULA 21 – VALIDAÇÃO LEAVE-ONE-GROUP-OUT (POR CONFIGURAÇÃO) x HOLDOUT FIXO
# Grupo = configuração experimental única (as 7 variáveis independentes). As
# réplicas de uma mesma configuração já foram agregadas no pré-processamento
# (no pré-processamento): cada linha das planilhas por área já é uma configuração única,
# então o Leave-One-Group-Out por configuração equivale, na prática, a um
# Leave-One-Out sobre as linhas de cada área, mas avalia com TODAS as linhas
# como teste, uma de cada vez, em vez de depender de um único corte fixo de 20%.
# Reusa os hiperparâmetros já escolhidos por área na execução em lote (não repete o
# Grid Search a cada fold), para isolar o efeito da validação cruzada do efeito
# do tuning de hiperparâmetros.
# ============================================================
from sklearn.model_selection import LeaveOneGroupOut


def construir_dados_completos(caminho, cobertura_minima=COBERTURA_MINIMA_ALVO):
    """Repete o tratamento da execução em lote (alvos, padronização, codificação), mas
    devolve TODAS as linhas da planilha, sem separar treino/teste/verificação."""
    df = pd.read_csv(caminho)
    alvos, _ = identificar_alvos(df, VARIAVEIS_INDEPENDENTES)
    alvos, _ = filtrar_alvos_por_cobertura(df, alvos, cobertura_minima)
    tratado = df.dropna(subset=alvos)
    tratado = padronizar_categorias(tratado, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES)
    X_area, _ = codificar_ordinal(tratado, VARIAVEIS_INDEPENDENTES)
    Y_area = tratado[alvos]
    return X_area, Y_area, alvos


resultados_logo = []

for caminho, insumo in zip(PLANILHAS, INSUMOS_LOTE):
    X_area, Y_area, alvos = construir_dados_completos(caminho)

    grupos = np.arange(len(X_area))  # cada linha já é uma configuração única
    logo = LeaveOneGroupOut()
    params = insumo["busca"].best_params_

    previsoes_oof = np.empty(Y_area.shape, dtype=float)
    for indice_treino, indice_teste in logo.split(X_area, Y_area, groups=grupos):
        modelo_fold = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **params)
        modelo_fold.fit(X_area.iloc[indice_treino], Y_area.iloc[indice_treino])
        previsoes_oof[indice_teste] = modelo_fold.predict(X_area.iloc[indice_teste])

    metricas_holdout = pd.DataFrame(
        avaliar_multioutput(insumo["Y_te"], insumo["previsoes_teste"], "Holdout", insumo["alvos"]))
    metricas_logo = pd.DataFrame(avaliar_multioutput(Y_area, previsoes_oof, "LOGO", alvos))

    erro_pct_holdout = np.abs((insumo["Y_te"].values - insumo["previsoes_teste"]) / insumo["Y_te"].values)
    erro_pct_logo = np.abs((Y_area.values - previsoes_oof) / Y_area.values)
    mape_holdout = erro_pct_holdout.mean() * 100
    mape_logo = erro_pct_logo.mean() * 100

    resultados_logo.append({
        "planilha":              insumo["nome_area"].replace("_data.csv", ""),
        "linhas_holdout_teste":  len(insumo["Y_te"]),
        "linhas_logo":           len(X_area),
        "MAE_holdout":           metricas_holdout["MAE"].mean(),
        "MAE_logo":              metricas_logo["MAE"].mean(),
        "RMSE_holdout":          metricas_holdout["RMSE"].mean(),
        "RMSE_logo":             metricas_logo["RMSE"].mean(),
        "R2_holdout":            metricas_holdout["R2"].mean(),
        "R2_logo":               metricas_logo["R2"].mean(),
        "MAPE_holdout":          mape_holdout,
        "MAPE_logo":             mape_logo,
        "dentro_de_10%_holdout": (erro_pct_holdout <= 0.10).mean(),
        "dentro_de_10%_logo":    (erro_pct_logo <= 0.10).mean(),
    })

comparativo_logo = pd.DataFrame(resultados_logo)

print("=" * 130)
print("HOLDOUT FIXO (70/20/10, split único, mesma seed) x LEAVE-ONE-GROUP-OUT (por configuração, todas as linhas)")
print("=" * 130)
print(comparativo_logo.round(4).to_string(index=False))

print("\nMédia entre as sete áreas:")
print(comparativo_logo[[
    "MAE_holdout", "MAE_logo", "RMSE_holdout", "RMSE_logo",
    "R2_holdout", "R2_logo", "MAPE_holdout", "MAPE_logo",
    "dentro_de_10%_holdout", "dentro_de_10%_logo",
]].mean().round(4).to_string())

## Célula 21b: Leave-One-Group-Out combinado com o estimador híbrido

A validação LOGO da célula anterior reavalia os modelos (7 variáveis, com os
mesmos hiperparâmetros escolhidos no tuning) trocando o holdout fixo por
Leave-One-Group-Out, mas ainda usa a previsão padrão do `RandomForestRegressor`
(`.predict()`, média das árvores).

Esta célula une as duas melhorias já validadas separadamente nas seções
anteriores: em cada fold do LOGO, a previsão passa a usar `prever_hibrido`
(célula 20 — mediana da folha quando a média cai fora do IQR, média caso
contrário) em vez de `.predict()`. Os hiperparâmetros continuam sendo os do
tuning; muda apenas a validação (LOGO em vez de holdout) e o estimador dentro
de cada fold (híbrido em vez de média padrão).

Como o LOGO já treina um modelo por linha (equivalente a Leave-One-Out nesta
base), e o híbrido percorre árvore por árvore dentro de cada fold, esta célula
é sensivelmente mais lenta que a validação LOGO simples.

In [ ]:
# ============================================================
# CÉLULA 21b – LEAVE-ONE-GROUP-OUT COM O ESTIMADOR HÍBRIDO
# Mesma validação LOGO da célula anterior (grupo = linha, na prática
# Leave-One-Out), mas cada fold prevê com `prever_hibrido` (célula 20:
# mediana da folha fora do IQR, média dentro) em vez de `.predict()`.
# Reusa os hiperparâmetros já escolhidos por área no tuning.
# ============================================================
resultados_logo_hibrido = []

for caminho, insumo in zip(PLANILHAS, INSUMOS_LOTE):
    X_area, Y_area, alvos = construir_dados_completos(caminho)

    grupos = np.arange(len(X_area))  # cada linha já é uma configuração única
    logo = LeaveOneGroupOut()
    params = insumo["busca"].best_params_

    previsoes_oof_hibrido = np.empty(Y_area.shape, dtype=float)
    for indice_treino, indice_teste in logo.split(X_area, Y_area, groups=grupos):
        X_tr_fold = X_area.iloc[indice_treino]
        Y_tr_fold = Y_area.iloc[indice_treino]
        X_te_fold = X_area.iloc[indice_teste]

        modelo_fold = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **params)
        modelo_fold.fit(X_tr_fold, Y_tr_fold)
        previsoes_oof_hibrido[indice_teste] = prever_hibrido(
            modelo_fold, X_tr_fold, Y_tr_fold, X_te_fold)

    metricas_logo_hibrido = pd.DataFrame(
        avaliar_multioutput(Y_area, previsoes_oof_hibrido, "LOGO_hibrido", alvos))

    erro_pct_logo_hibrido = np.abs(
        (Y_area.values - previsoes_oof_hibrido) / Y_area.values)

    resultados_logo_hibrido.append({
        "planilha":                  insumo["nome_area"].replace("_data.csv", ""),
        "linhas_logo":                len(X_area),
        "MAE_logo_hibrido":           metricas_logo_hibrido["MAE"].mean(),
        "RMSE_logo_hibrido":          metricas_logo_hibrido["RMSE"].mean(),
        "R2_logo_hibrido":            metricas_logo_hibrido["R2"].mean(),
        "MAPE_logo_hibrido":          metricas_logo_hibrido["MAPE_%"].mean(),
        "dentro_de_10%_logo_hibrido": (erro_pct_logo_hibrido <= 0.10).mean(),
    })

comparativo_logo_hibrido = pd.DataFrame(resultados_logo_hibrido)

print("=" * 120)
print("LEAVE-ONE-GROUP-OUT COM ESTIMADOR HÍBRIDO (7 variáveis, hiperparâmetros do tuning)")
print("=" * 120)
print(comparativo_logo_hibrido.round(4).to_string(index=False))

print("\nMédia entre as sete áreas:")
print(comparativo_logo_hibrido[[
    "MAE_logo_hibrido", "RMSE_logo_hibrido", "R2_logo_hibrido",
    "MAPE_logo_hibrido", "dentro_de_10%_logo_hibrido",
]].mean().round(4).to_string())

# Junta com o LOGO padrão (célula anterior) para comparar as duas versões do LOGO.
comparativo_logo = comparativo_logo.merge(comparativo_logo_hibrido, on="planilha")
melhorou_mae = int((comparativo_logo["MAE_logo_hibrido"] < comparativo_logo["MAE_logo"]).sum())
print(f"\nLOGO+híbrido melhora o MAE frente ao LOGO padrão em "
      f"{melhorou_mae} de {len(comparativo_logo)} áreas")


## Célula 22: Gráfico, holdout fixo x LOGO, por área

MAE, RMSE e R² lado a lado para as sete áreas.

In [ ]:
# ============================================================
# CÉLULA 22 – GRÁFICO: HOLDOUT FIXO x LEAVE-ONE-GROUP-OUT + HÍBRIDO, POR ÁREA
# Compara MAE, MAPE e R² lado a lado. A série "LOGO" do gráfico original foi
# substituída por "LOGO + híbrido" (célula 21b: validação Leave-One-Group-Out
# usando `prever_hibrido` em vez de `.predict()` em cada fold), mantendo o
# holdout fixo (split 70/20/10) como referência. O R² do holdout tem um
# outlier extremo em `mainback` (-32.98): o eixo é limitado para não esmagar
# as outras barras, e o valor real fica anotado na barra truncada.
# Paleta: mesmos dois primeiros slots categóricos (azul/laranja) já usados no
# gráfico de quantidade de árvores e no gráfico de previsto x medido, para as
# sete áreas.
# ============================================================
COR_HOLDOUT = "#2a78d6"
COR_LOGO    = "#eb6834"

areas_grafico = comparativo_logo["planilha"].tolist()
posicoes = np.arange(len(areas_grafico))
largura = 0.36

fig, eixos = plt.subplots(1, 3, figsize=(18, 5.5))


def barras_pareadas(eixo, coluna_holdout, coluna_logo, titulo, unidade, ylim=None):
    valores_holdout = comparativo_logo[coluna_holdout].values
    valores_logo = comparativo_logo[coluna_logo].values

    valores_plotados = np.clip(valores_holdout, ylim[0], None) if ylim else valores_holdout
    barras_h = eixo.bar(posicoes - largura / 2, valores_plotados, largura,
                         label="Holdout (split fixo)", color=COR_HOLDOUT, zorder=3)
    barras_l = eixo.bar(posicoes + largura / 2, valores_logo, largura,
                         label="LOGO + híbrido (todas as linhas)", color=COR_LOGO, zorder=3)

    for barra, valor in zip(barras_l, valores_logo):
        eixo.text(barra.get_x() + barra.get_width() / 2, valor, f"{valor:.2f}",
                   ha="center", va="bottom" if valor >= 0 else "top", fontsize=8, color="#52514e")

    for barra, valor in zip(barras_h, valores_holdout):
        if ylim and valor < ylim[0]:
            eixo.text(barra.get_x() + barra.get_width() / 2, ylim[0] * 0.85, f"{valor:.1f}",
                      ha="center", va="bottom", fontsize=8, color="white", fontweight="bold")
        else:
            eixo.text(barra.get_x() + barra.get_width() / 2, barra.get_height(), f"{valor:.2f}",
                       ha="center", va="bottom" if valor >= 0 else "top", fontsize=8, color="#52514e")

    eixo.set_title(titulo, fontsize=12, color="#0b0b0b")
    eixo.set_xticks(posicoes)
    eixo.set_xticklabels(areas_grafico, rotation=35, ha="right", fontsize=9)
    eixo.set_ylabel(unidade, fontsize=9, color="#52514e")
    eixo.axhline(0, color="#c3c2b7", linewidth=0.8, zorder=2)
    eixo.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    eixo.set_axisbelow(True)
    for spine in ["top", "right"]:
        eixo.spines[spine].set_visible(False)
    if ylim:
        eixo.set_ylim(ylim)


barras_pareadas(eixos[0], "MAE_holdout", "MAE_logo_hibrido", "MAE por área", "°C")
barras_pareadas(eixos[1], "MAPE_holdout", "MAPE_logo_hibrido", "MAPE por área", "%")
barras_pareadas(eixos[2], "R2_holdout", "R2_logo_hibrido", "R² por área", "", ylim=(-1.5, 1.0))

eixos[2].text(0.02, 0.96, "Barra truncada, valor real:\nmainback holdout = -32.98",
              transform=eixos[2].transAxes, fontsize=7.5, color="#898781", va="top")

handles, labels = eixos[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.97))
fig.suptitle("Holdout fixo (70/20/10) x Leave-One-Group-Out + híbrido, por área", fontsize=13, y=1.02, color="#0b0b0b")

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

## Célula 23: Gráfico, LOGO + híbrido x primeiro modelo (sem tuning), por área

Mesmo formato do gráfico anterior (MAE, MAPE e R² lado a lado), agora
comparando as duas melhorias combinadas na célula 21b — validação
Leave-One-Group-Out com o estimador híbrido, 7 variáveis, hiperparâmetros do
tuning — contra o ponto de partida do notebook: o `RandomForestRegressor` com
hiperparâmetros padrão do sklearn, sem Grid Search, avaliado no holdout fixo
(coluna `*_teste_padrao` da célula de registro do tuning).

É a distância entre o primeiro modelo treinado neste notebook e o resultado
depois de todas as melhorias testadas até aqui, isoladas nas seções
anteriores e reunidas aqui numa única comparação.

In [ ]:
# ============================================================
# CÉLULA 23 – GRÁFICO: LOGO + HÍBRIDO x PRIMEIRO MODELO (SEM TUNING), POR ÁREA
# Compara MAE, MAPE e R² lado a lado. "Sem tuning" é o RandomForestRegressor
# com hiperparâmetros padrão do sklearn (colunas *_teste_padrao de `tuning`,
# célula de registro do Grid Search), avaliado no holdout fixo. "LOGO +
# híbrido" vem de `comparativo_logo` (célula 21b). Mesma função
# `barras_pareadas` já definida no gráfico anterior.
# ============================================================
base_comparacao = comparativo_logo.merge(
    tuning[["key_area", "MAE_teste_padrao", "R2_teste_padrao", "MAPE_teste_padrao"]]
        .assign(planilha=lambda d: d["key_area"].str.replace("_data.csv", "", regex=False)),
    on="planilha",
)

areas_grafico2 = base_comparacao["planilha"].tolist()
posicoes = np.arange(len(areas_grafico2))

fig, eixos = plt.subplots(1, 3, figsize=(18, 5.5))

COR_SEM_TUNING = "#B0B0B0"
COR_LOGO_HIBRIDO = "#1baf7a"


def barras_pareadas_final(eixo, coluna_sem_tuning, coluna_logo_hibrido, titulo, unidade, ylim=None):
    valores_sem_tuning = base_comparacao[coluna_sem_tuning].values
    valores_logo_hibrido = base_comparacao[coluna_logo_hibrido].values

    valores_plotados = np.clip(valores_sem_tuning, ylim[0], None) if ylim else valores_sem_tuning
    barras_s = eixo.bar(posicoes - largura / 2, valores_plotados, largura,
                         label="Sem tuning (1o modelo)", color=COR_SEM_TUNING, zorder=3)
    barras_h = eixo.bar(posicoes + largura / 2, valores_logo_hibrido, largura,
                         label="LOGO + híbrido", color=COR_LOGO_HIBRIDO, zorder=3)

    for barra, valor in zip(barras_h, valores_logo_hibrido):
        eixo.text(barra.get_x() + barra.get_width() / 2, valor, f"{valor:.2f}",
                   ha="center", va="bottom" if valor >= 0 else "top", fontsize=8, color="#52514e")

    for barra, valor in zip(barras_s, valores_sem_tuning):
        if ylim and valor < ylim[0]:
            eixo.text(barra.get_x() + barra.get_width() / 2, ylim[0] * 0.85, f"{valor:.1f}",
                      ha="center", va="bottom", fontsize=8, color="white", fontweight="bold")
        else:
            eixo.text(barra.get_x() + barra.get_width() / 2, barra.get_height(), f"{valor:.2f}",
                       ha="center", va="bottom" if valor >= 0 else "top", fontsize=8, color="#52514e")

    eixo.set_title(titulo, fontsize=12, color="#0b0b0b")
    eixo.set_xticks(posicoes)
    eixo.set_xticklabels(areas_grafico2, rotation=35, ha="right", fontsize=9)
    eixo.set_ylabel(unidade, fontsize=9, color="#52514e")
    eixo.axhline(0, color="#c3c2b7", linewidth=0.8, zorder=2)
    eixo.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    eixo.set_axisbelow(True)
    for spine in ["top", "right"]:
        eixo.spines[spine].set_visible(False)
    if ylim:
        eixo.set_ylim(ylim)


barras_pareadas_final(eixos[0], "MAE_teste_padrao", "MAE_logo_hibrido", "MAE por área", "°C")
barras_pareadas_final(eixos[1], "MAPE_teste_padrao", "MAPE_logo_hibrido", "MAPE por área", "%")
barras_pareadas_final(eixos[2], "R2_teste_padrao", "R2_logo_hibrido", "R² por área", "", ylim=(-1.5, 1.0))

handles, labels = eixos[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, fontsize=10, bbox_to_anchor=(0.5, 0.97))
fig.suptitle("Primeiro modelo (sem tuning) x LOGO + híbrido, por área", fontsize=13, y=1.02, color="#0b0b0b")

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

melhorou_mae_final = int((base_comparacao["MAE_logo_hibrido"] < base_comparacao["MAE_teste_padrao"]).sum())
melhorou_r2_final = int((base_comparacao["R2_logo_hibrido"] > base_comparacao["R2_teste_padrao"]).sum())
print(f"MAE melhor com LOGO + híbrido em {melhorou_mae_final} de {len(base_comparacao)} áreas "
      "frente ao primeiro modelo (sem tuning)")
print(f"R² melhor com LOGO + híbrido em {melhorou_r2_final} de {len(base_comparacao)} áreas "
      "frente ao primeiro modelo (sem tuning)")


## Célula 24: Viés direcional e taxa de falso positivo, por área e por modelo

Os gráficos das Células 22 e 23 comparam MAE, MAPE e R² — métricas
simétricas, que tratam superestimar e subestimar a mesma temperatura como o
mesmo erro. Para esta aplicação isso é uma simplificação perigosa: um
termopar subestimado pode liberar como segura uma configuração que na
prática excede o limite de engenharia (falso negativo, o mais custoso em
segurança), enquanto superestimar gera um alerta desnecessário (falso
positivo, o mais custoso em operação — retrabalho, replanejamento de
ensaio). O MAE agregado não distingue as duas coisas.

Esta célula decompõe o erro (`previsto - medido`) por sinal e cruza cada
previsão com o limite de engenharia (`active_limit_c`) do
`sensor_registry.json` (usado pelo dashboard/frontend), para contar falsos
positivos e falsos negativos por área, comparando o primeiro modelo (sem
tuning, holdout) com o modelo final (LOGO + híbrido).

In [ ]:
# ============================================================
# CÉLULA 24 – VIÉS DIRECIONAL E TAXA DE FALSO POSITIVO/NEGATIVO
# Cruza as previsões (sem tuning x LOGO+híbrido) com active_limit_c do
# sensor_registry.json (frontend) para contar, por área:
#   - viés médio direcional (previsto - medido)
#   - falso positivo:  previsto > limite  e  medido <= limite  (alerta indevido)
#   - falso negativo:  previsto <= limite e  medido > limite   (risco não detectado)
# ATENÇÃO: active_limit_c é um placeholder de demo, não um limite de
# engenharia validado pela Whirlpool (ver aviso na célula anterior).
#
# Dois cuidados de correspondência de nomes, descobertos ao rodar esta célula:
#   1) 9 dos 77 sensores do registro têm active_limit_c=null porque são
#      "is_monitor": true (monitoram, mas não têm limite de engenharia ativo).
#      Não são falha de correspondência: são sensores sem limite por
#      definifição, e ficam de fora da contagem de FP/FN (não dá pra violar
#      um limite que não existe). Reportados à parte para não confundir com
#      nomes que simplesmente não bateram.
#   2) O nome da coluna na planilha de ipc_cooling tem um espaço a mais
#      ("IPC Housing Surface ") do que o tc_name do registro ("IPC Housing
#      Surface") -- normalizado aqui com .strip() antes do cruzamento.
# ============================================================
import json as _json

CAMINHO_REGISTRO_SENSORES = "../src/backend/data/sensor_registry.json"

with open(CAMINHO_REGISTRO_SENSORES, encoding="utf-8") as f:
    _registro_sensores = _json.load(f)

limite_por_sensor = {
    (registro["key_area"], registro["tc_name"].strip()): registro["active_limit_c"]
    for registro in _registro_sensores
}
sensores_sem_registro = {
    (registro["key_area"], registro["tc_name"].strip())
    for registro in _registro_sensores
    if registro["active_limit_c"] is None
}

NOME_AREA_PARA_KEY_AREA = {
    "mainback":      "Mainback",
    "coil_temps":    "Coil Temps",
    "main_vent_sys": "Main Vent Sys",
    "ct_knobs":      "CT Knobs",
    "ipc_comp":      "IPC Comp",
    "ipc_cooling":   "IPC Cooling",
    "oven_hmi":      "Oven HMI",
}

termopares_fora_do_registro = []
termopares_sem_limite_ativo = []


def classificar_fp_fn(y_real, y_previsto, alvos, planilha):
    # Viés direcional e contagem de FP/FN por termopar, para uma área.
    key_area = NOME_AREA_PARA_KEY_AREA[planilha]
    linhas = []
    y_real_arr = np.asarray(y_real, dtype=float)
    y_previsto_arr = np.asarray(y_previsto, dtype=float)

    for j, termopar in enumerate(alvos):
        chave = (key_area, termopar.strip())

        if chave not in limite_por_sensor:
            termopares_fora_do_registro.append((planilha, termopar))
            continue
        if chave in sensores_sem_registro:
            termopares_sem_limite_ativo.append((planilha, termopar))
            continue

        limite = limite_por_sensor[chave]
        real = y_real_arr[:, j]
        prev = y_previsto_arr[:, j]
        ok = ~np.isnan(real) & ~np.isnan(prev)
        if ok.sum() == 0:
            continue

        real_ok, prev_ok = real[ok], prev[ok]
        excede_real = real_ok > limite
        excede_prev = prev_ok > limite

        falso_positivo = (~excede_real) & excede_prev
        falso_negativo = excede_real & (~excede_prev)

        linhas.append({
            "planilha": planilha,
            "tc_name": termopar,
            "limite_C": limite,
            "n": int(ok.sum()),
            "vies_medio_C": float((prev_ok - real_ok).mean()),
            "n_falso_positivo": int(falso_positivo.sum()),
            "n_falso_negativo": int(falso_negativo.sum()),
            "taxa_falso_positivo": float(falso_positivo.mean()),
            "taxa_falso_negativo": float(falso_negativo.mean()),
        })

    return linhas


linhas_fp_sem_tuning = []
linhas_fp_logo_hibrido = []

for insumo in INSUMOS_LOTE:
    planilha = insumo["nome_area"].replace("_data.csv", "")

    linhas_fp_sem_tuning += classificar_fp_fn(
        insumo["Y_te"], insumo["previsoes_teste"], insumo["alvos"], planilha)

for caminho, insumo in zip(PLANILHAS, INSUMOS_LOTE):
    planilha = insumo["nome_area"].replace("_data.csv", "")
    X_area, Y_area, alvos = construir_dados_completos(caminho)

    grupos = np.arange(len(X_area))
    logo = LeaveOneGroupOut()
    params = insumo["busca"].best_params_

    previsoes_oof_hibrido_area = np.empty(Y_area.shape, dtype=float)
    for indice_treino, indice_teste in logo.split(X_area, Y_area, groups=grupos):
        X_tr_fold = X_area.iloc[indice_treino]
        Y_tr_fold = Y_area.iloc[indice_treino]
        X_te_fold = X_area.iloc[indice_teste]

        modelo_fold = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **params)
        modelo_fold.fit(X_tr_fold, Y_tr_fold)
        previsoes_oof_hibrido_area[indice_teste] = prever_hibrido(
            modelo_fold, X_tr_fold, Y_tr_fold, X_te_fold)

    linhas_fp_logo_hibrido += classificar_fp_fn(
        Y_area, previsoes_oof_hibrido_area, alvos, planilha)

fp_sem_tuning = pd.DataFrame(linhas_fp_sem_tuning)
fp_logo_hibrido = pd.DataFrame(linhas_fp_logo_hibrido)


def resumo_fp_por_area(tabela):
    return (tabela.groupby("planilha")
            .agg(n_termopares=("tc_name", "nunique"),
                 n_total=("n", "sum"),
                 vies_medio_C=("vies_medio_C", "mean"),
                 n_falso_positivo=("n_falso_positivo", "sum"),
                 n_falso_negativo=("n_falso_negativo", "sum"))
            .assign(taxa_falso_positivo=lambda d: d["n_falso_positivo"] / d["n_total"],
                    taxa_falso_negativo=lambda d: d["n_falso_negativo"] / d["n_total"])
            .reset_index())


resumo_sem_tuning = resumo_fp_por_area(fp_sem_tuning).set_index("planilha")
resumo_logo_hibrido = resumo_fp_por_area(fp_logo_hibrido).set_index("planilha")

comparativo_fp = resumo_sem_tuning[["vies_medio_C", "n_falso_positivo", "taxa_falso_positivo",
                                     "n_falso_negativo", "taxa_falso_negativo"]].join(
    resumo_logo_hibrido[["vies_medio_C", "n_falso_positivo", "taxa_falso_positivo",
                          "n_falso_negativo", "taxa_falso_negativo"]],
    lsuffix="_sem_tuning", rsuffix="_logo_hibrido",
).round(4)

print("=" * 130)
print("VIÉS DIRECIONAL E TAXA DE FALSO POSITIVO/NEGATIVO, POR ÁREA")
print("Limite = active_limit_c (sensor_registry.json) — PLACEHOLDER, não validado pela Whirlpool")
print("=" * 130)
print(comparativo_fp.to_string())

total_fp_sem_tuning = int(fp_sem_tuning["n_falso_positivo"].sum())
total_fp_logo_hibrido = int(fp_logo_hibrido["n_falso_positivo"].sum())
total_fn_sem_tuning = int(fp_sem_tuning["n_falso_negativo"].sum())
total_fn_logo_hibrido = int(fp_logo_hibrido["n_falso_negativo"].sum())

print(f"\nTotal de falsos positivos: sem tuning = {total_fp_sem_tuning}, "
      f"LOGO+híbrido = {total_fp_logo_hibrido}")
print(f"Total de falsos negativos: sem tuning = {total_fn_sem_tuning}, "
      f"LOGO+híbrido = {total_fn_logo_hibrido}")

areas_fp_melhorou = int((comparativo_fp["taxa_falso_positivo_logo_hibrido"]
                          < comparativo_fp["taxa_falso_positivo_sem_tuning"]).sum())
print(f"\nTaxa de falso positivo menor com LOGO+híbrido em {areas_fp_melhorou} "
      f"de {len(comparativo_fp)} áreas")

n_sem_limite = len(set(termopares_sem_limite_ativo))
n_fora_registro = len(set(termopares_fora_do_registro))
if n_sem_limite:
    print(f"\n{n_sem_limite} termopar(es) sem limite ativo no registro "
          "(is_monitor=true, active_limit_c=null) — fora da contagem de FP/FN:")
    for planilha, termopar in sorted(set(termopares_sem_limite_ativo)):
        print(f"   {planilha:15s} {termopar}")
if n_fora_registro:
    print(f"\nATENÇÃO — {n_fora_registro} termopar(es) do modelo sem correspondência "
          "no sensor_registry.json (nome pode ter mudado):")
    for planilha, termopar in sorted(set(termopares_fora_do_registro)):
        print(f"   {planilha:15s} {termopar!r}")


## 19. Experimento: modelo por `run_case` (só `coil_temps_data.csv`)

**Escopo desta seção: experimento de análise, não muda o modelo em produção.** A arquitetura vigente do projeto é um modelo único, sem `run_case` como parâmetro do dashboard (decisão de produto já registrada no WAD). O que segue serve só para entender se treinar um Random Forest separado por `run_case` reduziria o MAE de `coil_temps_data.csv` (8,93 °C no holdout, o pior das sete áreas — Seção 8), sem propor mudança de escopo do produto.

**Por que `coil` e por que `run_case`.** A importância de variáveis já calculada nas Seções 5 e 11 mostra `run_case` concentrando 0,42 a 0,84 da importância preditiva em `coil_temps_data.csv`, bem acima do observado nas outras áreas. Isso bate com a amplitude térmica real entre tipos de teste discutida no pré-processamento: `Pyro` chega a ~50 °C nas Coil temperatures, `Broil C2` a ~220 °C. A hipótese é que um único Random Forest pode estar tendo dificuldade em aprender essa mudança de patamar a partir de uma única variável categórica ordinalmente codificada, e que modelos separados por `run_case` isolariam esse salto.

**Limitação conhecida antes de começar.** `coil_temps_data.csv` tem 32 linhas no total. Fatiando por `run_case` (4 categorias), cada submodelo treina com uma fração disso — pouquíssimo para Random Forest. Por isso a avaliação aqui usa Leave-One-Out dentro de cada subconjunto (como na Seção 18), em vez de um holdout fixo, para não desperdiçar linhas só na medição. Ainda assim, o resultado deve ser lido como indicativo, não conclusivo — o risco R02/R03 (Seção 4.1.5 da documentação) piora, não melhora, ao fragmentar ainda mais uma base já pequena.

In [ ]:
# ============================================================
# CÉLULA 19a — MODELO POR run_case, SÓ coil_temps_data.csv
# Reusa as funções de tratamento já definidas nas Células 2–4. run_case sai
# das features (vira o critério de separação dos subconjuntos) e cada
# subconjunto é avaliado com Leave-One-Out, dado o tamanho pequeno de cada um.
# ============================================================
CAMINHO_COIL = "../data/coil_temps_data.csv"
VARIAVEIS_SEM_RUN_CASE = [v for v in VARIAVEIS_INDEPENDENTES if v != "run_case"]

dados_coil_bruto = pd.read_csv(CAMINHO_COIL)
alvos_coil, _ = identificar_alvos(dados_coil_bruto, VARIAVEIS_INDEPENDENTES)
alvos_coil, removidos_coil = filtrar_alvos_por_cobertura(
    dados_coil_bruto, alvos_coil, COBERTURA_MINIMA_ALVO
)

coil_tratado = dados_coil_bruto.dropna(subset=alvos_coil).copy()
coil_tratado = padronizar_categorias(coil_tratado, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES)

print(f"Linhas de coil após tratamento: {len(coil_tratado)}")
print(f"Alvos (termopares): {len(alvos_coil)}")
print("\nDistribuição de run_case:")
print(coil_tratado["run_case"].value_counts().to_string())


In [ ]:
# ============================================================
# CÉLULA 19b — TREINO E AVALIAÇÃO (LEAVE-ONE-OUT) POR run_case
# Um RandomForestRegressor por valor de run_case, hiperparâmetros iguais aos
# já escolhidos para coil_temps_data.csv no tuning (Célula 10 / execução em
# lote), para isolar o efeito da segmentação do efeito do tuning.
# ============================================================
from sklearn.model_selection import LeaveOneOut

insumo_coil = next(i for i in INSUMOS_LOTE if i["nome_area"] == "coil_temps_data.csv")
PARAMS_COIL = insumo_coil["busca"].best_params_
print(f"Hiperparâmetros reaproveitados do tuning de coil: {PARAMS_COIL}\n")

resultados_por_run_case = []
previsoes_oof_por_run_case = {}

for caso in sorted(coil_tratado["run_case"].unique()):
    subconjunto = coil_tratado[coil_tratado["run_case"] == caso]
    X_sub, _ = codificar_ordinal(subconjunto, VARIAVEIS_SEM_RUN_CASE)
    Y_sub = subconjunto[alvos_coil]

    if len(X_sub) < 4:
        print(f"{caso!r}: apenas {len(X_sub)} linhas, pulado (Leave-One-Out exige mais dados).")
        continue

    loo = LeaveOneOut()
    previsoes_oof = np.empty(Y_sub.shape, dtype=float)
    for indice_treino, indice_teste in loo.split(X_sub):
        modelo_fold = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **PARAMS_COIL)
        modelo_fold.fit(X_sub.iloc[indice_treino], Y_sub.iloc[indice_treino])
        previsoes_oof[indice_teste] = modelo_fold.predict(X_sub.iloc[indice_teste])

    previsoes_oof_por_run_case[caso] = (Y_sub, previsoes_oof)
    metricas_caso = pd.DataFrame(avaliar_multioutput(Y_sub, previsoes_oof, "LOO", alvos_coil))
    erro_pct_caso = np.abs((Y_sub.values - previsoes_oof) / Y_sub.values)

    resultados_por_run_case.append({
        "run_case": caso,
        "linhas": len(X_sub),
        "MAE_loo": metricas_caso["MAE"].mean(),
        "RMSE_loo": metricas_caso["RMSE"].mean(),
        "R2_loo": metricas_caso["R2"].mean(),
        "MAPE_loo_%": metricas_caso["MAPE_%"].mean(),
        "dentro_de_10%": (erro_pct_caso <= 0.10).mean(),
    })

comparativo_run_case = pd.DataFrame(resultados_por_run_case)
print("=" * 100)
print("MODELO SEPARADO POR run_case (coil_temps_data.csv), Leave-One-Out dentro de cada grupo")
print("=" * 100)
print(comparativo_run_case.round(4).to_string(index=False))


In [ ]:
# ============================================================
# CÉLULA 19c — MAE AGREGADO (PONDERADO POR LINHAS) x MODELO ÚNICO
# Junta as previsões fora-da-amostra de todos os run_case testados acima num
# único MAE, para comparar com o LOGO do modelo único (Célula 21, variável
# comparativo_logo) na mesma unidade e no mesmo tipo de validação
# (fora-da-amostra, uma linha de cada vez).
# ============================================================
Y_concatenado = pd.concat([y for y, _ in previsoes_oof_por_run_case.values()], axis=0)
previsoes_concatenadas = np.concatenate([p for _, p in previsoes_oof_por_run_case.values()], axis=0)

mae_agregado_run_case = mean_absolute_error(Y_concatenado.values, previsoes_concatenadas)
rmse_agregado_run_case = np.sqrt(mean_squared_error(Y_concatenado.values, previsoes_concatenadas))

mae_modelo_unico_logo = comparativo_logo.loc[
    comparativo_logo["planilha"] == "coil_temps", "MAE_logo"
].iloc[0]
rmse_modelo_unico_logo = comparativo_logo.loc[
    comparativo_logo["planilha"] == "coil_temps", "RMSE_logo"
].iloc[0]

print("=" * 90)
print("COMPARAÇÃO FINAL: MODELO ÚNICO (LOGO) x MODELOS POR run_case (LOO agregado)")
print("=" * 90)
print(f"Linhas usadas na comparação: {len(Y_concatenado)} de {len(coil_tratado)} "
      f"(run_case com menos de 4 linhas ficaram de fora)\n")
print(f"Modelo único   | MAE = {mae_modelo_unico_logo:6.3f} °C | RMSE = {rmse_modelo_unico_logo:6.3f} °C")
print(f"Por run_case   | MAE = {mae_agregado_run_case:6.3f} °C | RMSE = {rmse_agregado_run_case:6.3f} °C")

diferenca_mae = mae_modelo_unico_logo - mae_agregado_run_case
sinal = "melhor" if diferenca_mae > 0 else "pior"
print(f"\nSegmentar por run_case teve MAE {abs(diferenca_mae):.3f} °C {sinal} "
      "que o modelo único nesta comparação.")


In [ ]:
# ============================================================
# CÉLULA 19d — COMPARAÇÃO JUSTA: MODELO ÚNICO NAS MESMAS 27 LINHAS
# A célula anterior compara o modelo por run_case (só Broil C2 e Broil
# Qualitative, 27 linhas) com o LOGO do modelo único rodado nas 32 linhas de
# coil (incluindo Pyro e Broil C1, os casos mais difíceis). Para isolar o
# efeito de segmentar por run_case do efeito de qual subconjunto é avaliado,
# esta célula recalcula o LOGO do modelo único restrito às mesmas 27 linhas.
# ============================================================
from sklearn.model_selection import LeaveOneGroupOut

linhas_comparaveis = coil_tratado["run_case"].isin(previsoes_oof_por_run_case.keys())
coil_comparavel = coil_tratado[linhas_comparaveis]

X_comparavel, _ = codificar_ordinal(coil_comparavel, VARIAVEIS_INDEPENDENTES)
Y_comparavel = coil_comparavel[alvos_coil]

logo_comparavel = LeaveOneGroupOut()
grupos_comparavel = np.arange(len(X_comparavel))
previsoes_oof_unico_comparavel = np.empty(Y_comparavel.shape, dtype=float)

for indice_treino, indice_teste in logo_comparavel.split(X_comparavel, groups=grupos_comparavel):
    modelo_fold = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **PARAMS_COIL)
    modelo_fold.fit(X_comparavel.iloc[indice_treino], Y_comparavel.iloc[indice_treino])
    previsoes_oof_unico_comparavel[indice_teste] = modelo_fold.predict(X_comparavel.iloc[indice_teste])

mae_modelo_unico_comparavel = mean_absolute_error(Y_comparavel.values, previsoes_oof_unico_comparavel)
rmse_modelo_unico_comparavel = np.sqrt(mean_squared_error(Y_comparavel.values, previsoes_oof_unico_comparavel))

print("=" * 90)
print("COMPARAÇÃO JUSTA: MESMAS 27 LINHAS (Broil C2 + Broil Qualitative) NOS DOIS MODELOS")
print("=" * 90)
print(f"Modelo único   (27 linhas) | MAE = {mae_modelo_unico_comparavel:6.3f} °C | "
      f"RMSE = {rmse_modelo_unico_comparavel:6.3f} °C")
print(f"Por run_case   (27 linhas) | MAE = {mae_agregado_run_case:6.3f} °C | "
      f"RMSE = {rmse_agregado_run_case:6.3f} °C")

diferenca_justa = mae_modelo_unico_comparavel - mae_agregado_run_case
sinal_justa = "melhor" if diferenca_justa > 0 else "pior"
print(f"\nNa mesma fatia de dados, segmentar por run_case teve MAE "
      f"{abs(diferenca_justa):.3f} °C {sinal_justa} que o modelo único.")


In [ ]:
# ============================================================
# GRÁFICO 1 — MAE e RMSE POR run_case (modelos separados, LOO)
# ============================================================
x = np.arange(len(comparativo_run_case))
largura = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
barras_mae = ax.bar(x - largura / 2, comparativo_run_case["MAE_loo"], largura,
                     label="MAE", color="#4C72B0")
barras_rmse = ax.bar(x + largura / 2, comparativo_run_case["RMSE_loo"], largura,
                      label="RMSE", color="#DD8452")

for barras in (barras_mae, barras_rmse):
    ax.bar_label(barras, fmt="%.2f", padding=2, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(comparativo_run_case["run_case"])
ax.set_ylabel("Erro (°C)")
ax.set_title("Coil: MAE e RMSE por run_case (modelo separado, Leave-One-Out)",
             fontsize=12, fontweight="bold")
ax.legend()
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# GRÁFICO 2 — MODELO ÚNICO x MODELO POR run_case: DUAS COMPARAÇÕES
# Painel esquerdo: como reportado na Célula 19c (32 linhas no modelo único x
# 27 no modelo por run_case). Painel direito: comparação justa da Célula 19d
# (mesmas 27 linhas nos dois modelos).
# ============================================================
fig, eixos = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

comparacoes = [
    (
        eixos[0],
        "Como reportado\n(único: 32 linhas | run_case: 27 linhas)",
        mae_modelo_unico_logo, rmse_modelo_unico_logo,
        mae_agregado_run_case, rmse_agregado_run_case,
    ),
    (
        eixos[1],
        "Comparação justa\n(mesmas 27 linhas nos dois)",
        mae_modelo_unico_comparavel, rmse_modelo_unico_comparavel,
        mae_agregado_run_case, rmse_agregado_run_case,
    ),
]

largura = 0.35
x = np.arange(2)  # MAE, RMSE

for ax, titulo, mae_unico, rmse_unico, mae_rc, rmse_rc in comparacoes:
    valores_unico = [mae_unico, rmse_unico]
    valores_run_case = [mae_rc, rmse_rc]

    barras_unico = ax.bar(x - largura / 2, valores_unico, largura,
                           label="Modelo único", color="#B0B0B0")
    barras_rc = ax.bar(x + largura / 2, valores_run_case, largura,
                        label="Por run_case", color="#4C72B0")

    for barras in (barras_unico, barras_rc):
        ax.bar_label(barras, fmt="%.2f", padding=2, fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(["MAE", "RMSE"])
    ax.set_title(titulo, fontsize=10)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)

eixos[0].set_ylabel("Erro (°C)")
eixos[0].legend(loc="upper left")
fig.suptitle("Coil: modelo único x modelo por run_case", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### Leitura do resultado

O Gráfico 1 mostra o MAE e o RMSE de cada `run_case` avaliado isoladamente. O Gráfico 2 traz duas comparações lado a lado contra o modelo único:

- **Painel esquerdo (como reportado na Célula 19c):** modelo único avaliado nas 32 linhas de coil (inclui `Pyro` e `Broil C1`) contra o modelo por `run_case` avaliado só nas 27 linhas de `Broil C2`/`Broil Qualitative`. Favorece o modelo por `run_case`, porque ele nunca precisou prever os casos mais extremos.
- **Painel direito (comparação justa, Célula 19d):** os dois modelos avaliados nas mesmas 27 linhas. Essa é a comparação que isola o efeito de segmentar por `run_case` do efeito de qual subconjunto está sendo medido.

Leia o painel direito antes de tirar qualquer conclusão:

- **Se o modelo por `run_case` ainda for melhor no painel direito**, a hipótese de que uma árvore única tem dificuldade em aprender o salto entre regimes térmicos (`Pyro` ~50 °C vs. `Broil C2` ~220 °C) ganha evidência dentro dos casos com dados suficientes, mas isso **não cobre** `Pyro` nem `Broil C1` (3 e 2 linhas, respectivamente) — exatamente os regimes mais extremos, onde faltam dados até para Leave-One-Out. O ganho continua sendo indicativo, não conclusivo.
- **Se a vantagem cair ou inverter no painel direito**, o ganho aparente do painel esquerdo era, ao menos em parte, efeito de comparar conjuntos de teste diferentes (um mais fácil que o outro), não da segmentação em si — sinal de que o caminho mais promissor é a alternativa mais barata cogitada, features de interação (`run_case` × `losses`, `run_case` × `deflector_hole`) dentro do modelo único, sem fatiar a amostra.

**Em qualquer cenário, isto não muda o modelo de produção.** A decisão de manter um modelo único sem `run_case` como input do dashboard já foi tomada fora deste notebook; este experimento serve para informar futuras discussões de modelagem, não para revertê-la.

## 20. Experimento: one-hot encoding + estimador híbrido (só `coil_temps_data.csv`)

**Escopo: experimento de análise, não muda o modelo em produção.** A Seção 19 testou segmentar `coil` por `run_case` e, na comparação justa (mesmas linhas nos dois modelos), o modelo único saiu levemente melhor (MAE 8,18 °C x 8,37 °C) — fragmentar os 32 registros em subconjuntos menores não compensou. Esta seção testa uma via diferente que preserva todas as linhas: manter um único modelo, mas trocar a **codificação ordinal** das 7 variáveis independentes (usada em todo o notebook desde a Célula 3) por **one-hot encoding**, combinada com o **estimador híbrido** (mediana nas caudas, média no meio — Célula 20/`prever_hibrido`).

**Por que isso pode importar para `coil`.** A codificação ordinal impõe uma ordem arbitrária entre categorias sem relação natural de grandeza (ex.: `run_case` = 0/1/2/3 para Broil C1/Broil C2/Broil Qualitative/Pyro, em ordem alfabética). Uma árvore de decisão só consegue separar essas categorias por cortes do tipo "valor ≤ X", o que pode exigir mais divisões para chegar ao mesmo particionamento que o one-hot alcança em um único corte por categoria. Como `run_case` é a variável dominante em `coil` (Seções 5 e 11), o formato da codificação dela tem mais chance de importar aqui do que nas outras áreas.

**O que esta seção mede.** Mesma validação (Leave-One-Group-Out, na prática Leave-One-Out sobre as 32 linhas) já usada nas Células 21 e 21b, agora no espaço one-hot com o estimador híbrido. O hiperparâmetro é re-buscado via `GridSearchCV`, porque o número de colunas de entrada muda (7 ordinais → mais colunas binárias no one-hot) e reaproveitar os hiperparâmetros escolhidos para o espaço ordinal não seria uma comparação válida.


In [ ]:
# ============================================================
# CÉLULA 20a — ONE-HOT ENCODING DE coil_temps_data.csv
# Reusa o mesmo tratamento de nulos/grafias da Célula 19a (identificar_alvos,
# filtrar_alvos_por_cobertura, padronizar_categorias), mas troca a codificacao
# ordinal por one-hot (pd.get_dummies) nas 7 variaveis independentes.
# ============================================================
coil_onehot_base = dados_coil_bruto.dropna(subset=alvos_coil).copy()
coil_onehot_base = padronizar_categorias(coil_onehot_base, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES)

X_coil_onehot = pd.get_dummies(
    coil_onehot_base[VARIAVEIS_INDEPENDENTES], columns=VARIAVEIS_INDEPENDENTES
)
Y_coil_onehot = coil_onehot_base[alvos_coil]

print(f"Linhas: {len(X_coil_onehot)}")
print(f"Colunas após one-hot: {X_coil_onehot.shape[1]} (eram {len(VARIAVEIS_INDEPENDENTES)} ordinais)")
print(f"\nColunas geradas:\n{list(X_coil_onehot.columns)}")


In [ ]:
# ============================================================
# CÉLULA 20b — TUNING NO ESPAÇO ONE-HOT (mesma grade da Célula 5)
# Re-busca os hiperparametros porque a dimensionalidade da entrada mudou;
# usa a mesma GRADE_HIPERPARAMETROS e a mesma regra de folds da execucao em
# lote, para isolar o efeito da codificacao do efeito do tuning.
# ============================================================
n_folds_onehot = max(2, min(N_FOLDS_DESEJADO, int(len(X_coil_onehot) * PROPORCAO_TREINO)))

busca_onehot = GridSearchCV(
    estimator=RandomForestRegressor(random_state=SEMENTE, n_jobs=-1),
    param_grid=GRADE_HIPERPARAMETROS,
    cv=n_folds_onehot,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
).fit(X_coil_onehot, Y_coil_onehot)

PARAMS_COIL_ONEHOT = busca_onehot.best_params_
print(f"Melhores hiperparâmetros (espaço one-hot): {PARAMS_COIL_ONEHOT}")


In [ ]:
# ============================================================
# CÉLULA 20c — LOGO NO ESPAÇO ONE-HOT, ESTIMADOR HÍBRIDO
# Repete a mesma validacao das Celulas 21/21b (LeaveOneGroupOut, grupo = linha),
# desta vez no espaco one-hot, com prever_hibrido (Celula 20/estimador hibrido,
# ja definida e reutilizada aqui sem alteracao).
# ============================================================
grupos_onehot = np.arange(len(X_coil_onehot))
logo_onehot = LeaveOneGroupOut()

previsoes_oof_onehot_hibrido = np.empty(Y_coil_onehot.shape, dtype=float)

for indice_treino, indice_teste in logo_onehot.split(X_coil_onehot, groups=grupos_onehot):
    X_tr_fold = X_coil_onehot.iloc[indice_treino]
    Y_tr_fold = Y_coil_onehot.iloc[indice_treino]
    X_te_fold = X_coil_onehot.iloc[indice_teste]

    modelo_fold = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **PARAMS_COIL_ONEHOT)
    modelo_fold.fit(X_tr_fold, Y_tr_fold)

    previsoes_oof_onehot_hibrido[indice_teste] = prever_hibrido(
        modelo_fold, X_tr_fold, Y_tr_fold, X_te_fold)

metricas_onehot_hibrido = pd.DataFrame(
    avaliar_multioutput(Y_coil_onehot, previsoes_oof_onehot_hibrido, "LOGO_onehot_hibrido", alvos_coil))

mae_onehot_hibrido = metricas_onehot_hibrido["MAE"].mean()
rmse_onehot_hibrido = metricas_onehot_hibrido["RMSE"].mean()
r2_onehot_hibrido = metricas_onehot_hibrido["R2"].mean()

print("=" * 90)
print("LOGO NO ESPAÇO ONE-HOT + ESTIMADOR HÍBRIDO (coil_temps_data.csv)")
print("=" * 90)
print(f"One-hot + híbrido  | MAE = {mae_onehot_hibrido:6.3f} °C | RMSE = {rmse_onehot_hibrido:6.3f} °C | R² = {r2_onehot_hibrido:.3f}")


### Leitura do resultado

A validação acima aplica, para `coil_temps_data.csv`, one-hot encoding combinado ao estimador híbrido, avaliado por LOGO (equivalente a Leave-One-Out nesta base de 32 linhas).

**Isto não muda o modelo de produção.** É uma comprovação restrita a `coil_temps_data.csv`; a Seção 21 repete o mesmo experimento nas sete áreas antes de qualquer conclusão mais ampla.


## 21. Experimento: one-hot encoding + estimador híbrido nas sete áreas (LOGO)

**Escopo: extensão da Seção 20 para todas as `key_area`s.** A Seção 20 testou one-hot + estimador híbrido apenas em `coil_temps_data.csv`, porque `run_case` é a variável dominante lá e a ordem alfabética imposta pela codificação ordinal (0/1/2/3 para Broil C1/Broil C2/Broil Qualitative/Pyro) é arbitrária. O resultado de uma área só não diz se a combinação ajuda o modelo como um todo — esta seção repete o mesmo experimento nas sete planilhas.

**O que muda e o que fica fixo.** Fica fixo todo o pré-processamento das Células 3 e 7 (seleção de alvos por cobertura, descarte de linhas com alvo ausente, padronização de grafias, nulos como categoria explícita `Nao Informado`) e a validação (Leave-One-Group-Out com grupo = configuração, o que nesta base de 32 linhas por área equivale a Leave-One-Out). Muda a última etapa do pré-processamento (`pd.get_dummies` no lugar de `codificar_ordinal`), e a previsão passa a usar sempre o estimador híbrido da Célula 20.

**Por que o tuning é refeito.** O número de colunas de entrada deixa de ser 7 e passa a depender de quantas categorias cada área tem, então reaproveitar os hiperparâmetros escolhidos no espaço ordinal não seria comparação válida. A busca é refeita **nas mesmas linhas de treino (70%) que a Célula 7 sorteou para cada área**.

**Grade de tuning reduzida em relação à `GRADE_HIPERPARAMETROS` original.** O estimador híbrido (`prever_hibrido`, Célula 20) percorre árvore por árvore em Python puro, sem paralelismo; seu custo escala com `n_estimators`. Repetir esse laço em LOGO (32 folds por área) para as sete áreas, com `n_estimators` podendo chegar a 800 (teto da grade original), tornou a execução impraticável — a primeira tentativa foi interrompida após mais de 1h30 sem terminar. A Célula 25b usa uma grade menor, com teto de 300 árvores, só para esta seção.


In [ ]:
# ============================================================
# CÉLULA 25a — ONE-HOT ENCODING DAS SETE PLANILHAS
# Mesma sequencia de tratamento de `construir_dados_completos` (Celula 21):
# identificar_alvos -> filtrar_alvos_por_cobertura -> dropna nos alvos ->
# padronizar_categorias. So a ultima etapa muda: pd.get_dummies no lugar de
# codificar_ordinal. O one-hot e ajustado na planilha inteira, para que todos
# os folds do LOGO compartilhem o mesmo conjunto de colunas binarias.
# ============================================================
def construir_onehot_completo(caminho, cobertura_minima=COBERTURA_MINIMA_ALVO):
    """Versão one-hot de `construir_dados_completos`: mesmas linhas e alvos, features binárias."""
    df = pd.read_csv(caminho)
    alvos, _ = identificar_alvos(df, VARIAVEIS_INDEPENDENTES)
    alvos, _ = filtrar_alvos_por_cobertura(df, alvos, cobertura_minima)
    tratado = df.dropna(subset=alvos)
    tratado = padronizar_categorias(tratado, VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES)
    X_area = pd.get_dummies(tratado[VARIAVEIS_INDEPENDENTES], columns=VARIAVEIS_INDEPENDENTES)
    Y_area = tratado[alvos]
    return X_area, Y_area, alvos


DADOS_ONEHOT_POR_AREA = {}
linhas_dimensao = []

for caminho in PLANILHAS:
    X_oh, Y_oh, alvos_oh = construir_onehot_completo(caminho)
    nome_area = caminho.split("/")[-1]
    DADOS_ONEHOT_POR_AREA[nome_area] = (X_oh, Y_oh, alvos_oh)
    linhas_dimensao.append({
        "key_area":          nome_area.replace("_data.csv", ""),
        "linhas":            len(X_oh),
        "alvos":             len(alvos_oh),
        "colunas_ordinais":  len(VARIAVEIS_INDEPENDENTES),
        "colunas_onehot":    X_oh.shape[1],
        "linhas_por_coluna": len(X_oh) / X_oh.shape[1],
    })

dimensao_onehot = pd.DataFrame(linhas_dimensao)

print("=" * 90)
print("DIMENSÃO DA ENTRADA APÓS ONE-HOT, POR ÁREA")
print("=" * 90)
print(dimensao_onehot.round(2).to_string(index=False))
print("\nCom codificação ordinal são sempre 7 colunas e ~4,6 linhas por coluna;")
print("o one-hot multiplica as colunas e deixa menos de 1,5 linha por coluna em todas as áreas.")

In [ ]:
# ============================================================
# CÉLULA 25b — TUNING NO ESPAÇO ONE-HOT, ÁREA POR ÁREA (grade reduzida)
# A dimensionalidade da entrada muda com o one-hot, entao reaproveitar os
# hiperparametros escolhidos no espaco ordinal nao seria comparacao valida.
# Para a busca ficar simetrica ao baseline ordinal, ela usa exatamente as
# mesmas linhas de treino (70%) que a Celula 7 sorteou para cada area — os
# indices vem de INSUMOS_LOTE["X_tr"] — com a mesma regra de folds da
# execucao em lote.
#
# A grade e MENOR que GRADE_HIPERPARAMETROS (usada nas Celulas 5, 7 e 21/21b):
# prever_hibrido percorre arvore por arvore em Python puro, e seu custo
# escala com n_estimators. Repetir isso em LOGO (32 folds) para as sete
# areas com n_estimators ate 800 (teto da grade original) tornou a execucao
# impraticavel na primeira tentativa (>1h30 sem terminar). O teto aqui e 300,
# igual ao N_ARVORES_HIBRIDO ja usado na Celula 20.
# ============================================================
GRADE_HIPERPARAMETROS_ONEHOT = {
    "n_estimators":     [100, 300],
    "max_depth":        [5, 8, None],
    "min_samples_leaf": [1, 2],
    "max_features":     ["sqrt", None],
}

PARAMS_ONEHOT_POR_AREA = {}
linhas_tuning_onehot = []

for insumo in INSUMOS_LOTE:
    nome_area = insumo["nome_area"]
    X_oh, Y_oh, _ = DADOS_ONEHOT_POR_AREA[nome_area]

    indices_treino = insumo["X_tr"].index
    X_tr_oh = X_oh.loc[indices_treino]
    Y_tr_oh = Y_oh.loc[indices_treino]

    n_folds_area = max(2, min(N_FOLDS_DESEJADO, len(X_tr_oh)))
    busca_oh = GridSearchCV(
        estimator=RandomForestRegressor(random_state=SEMENTE, n_jobs=-1),
        param_grid=GRADE_HIPERPARAMETROS_ONEHOT,
        cv=n_folds_area,
        scoring="neg_mean_absolute_error",
        n_jobs=-1,
    ).fit(X_tr_oh, Y_tr_oh)

    PARAMS_ONEHOT_POR_AREA[nome_area] = busca_oh.best_params_
    linhas_tuning_onehot.append({
        "key_area":       nome_area.replace("_data.csv", ""),
        "linhas_treino":  len(X_tr_oh),
        "n_folds":        n_folds_area,
        "MAE_cv_onehot":  -busca_oh.best_score_,
        **{f"onehot_{chave}": valor for chave, valor in busca_oh.best_params_.items()},
    })

tuning_onehot = pd.DataFrame(linhas_tuning_onehot)

print("=" * 120)
print("HIPERPARÂMETROS ESCOLHIDOS NO ESPAÇO ONE-HOT (grade reduzida, mesmas linhas de treino do lote)")
print("=" * 120)
print(tuning_onehot.round(4).to_string(index=False))
print("\nMAE_cv_onehot é o erro da própria validação interna do GridSearchCV (não é resultado de teste).")


In [ ]:
# ============================================================
# CÉLULA 25c — LOGO NO ESPAÇO ONE-HOT NAS SETE ÁREAS, ESTIMADOR HÍBRIDO
# Repete a validacao das Celulas 21 e 21b (LeaveOneGroupOut, grupo = linha, na
# pratica Leave-One-Out sobre as 32 linhas de cada area), desta vez com as
# features one-hot, os hiperparametros da Celula 25b (grade reduzida, teto
# de 300 arvores) e prever_hibrido (Celula 20) como estimador.
# ============================================================
resultados_onehot_areas = []

for insumo in INSUMOS_LOTE:
    nome_area = insumo["nome_area"]
    X_oh, Y_oh, alvos_oh = DADOS_ONEHOT_POR_AREA[nome_area]
    params_oh = PARAMS_ONEHOT_POR_AREA[nome_area]

    grupos = np.arange(len(X_oh))
    logo = LeaveOneGroupOut()

    previsoes_oof_hibrido = np.empty(Y_oh.shape, dtype=float)

    for indice_treino, indice_teste in logo.split(X_oh, groups=grupos):
        X_tr_fold = X_oh.iloc[indice_treino]
        Y_tr_fold = Y_oh.iloc[indice_treino]
        X_te_fold = X_oh.iloc[indice_teste]

        modelo_fold = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **params_oh)
        modelo_fold.fit(X_tr_fold, Y_tr_fold)

        previsoes_oof_hibrido[indice_teste] = prever_hibrido(
            modelo_fold, X_tr_fold, Y_tr_fold, X_te_fold)

    metricas_hibrido = pd.DataFrame(
        avaliar_multioutput(Y_oh, previsoes_oof_hibrido, "LOGO_onehot_hibrido", alvos_oh))

    erro_pct_hibrido = np.abs((Y_oh.values - previsoes_oof_hibrido) / Y_oh.values)

    resultados_onehot_areas.append({
        "planilha":                          nome_area.replace("_data.csv", ""),
        "linhas_logo":                       len(X_oh),
        "colunas_onehot":                    X_oh.shape[1],
        "MAE_logo_onehot_hibrido":           metricas_hibrido["MAE"].mean(),
        "RMSE_logo_onehot_hibrido":          metricas_hibrido["RMSE"].mean(),
        "R2_logo_onehot_hibrido":            metricas_hibrido["R2"].mean(),
        "MAPE_logo_onehot_hibrido":          metricas_hibrido["MAPE_%"].mean(),
        "dentro_de_10%_logo_onehot_hibrido": (erro_pct_hibrido <= 0.10).mean(),
    })
    print(f"OK   {nome_area}")

comparativo_onehot_areas = pd.DataFrame(resultados_onehot_areas)

print("\n" + "=" * 110)
print("LEAVE-ONE-GROUP-OUT NO ESPAÇO ONE-HOT + ESTIMADOR HÍBRIDO, AS SETE ÁREAS")
print("=" * 110)
print(comparativo_onehot_areas.round(4).to_string(index=False))


In [ ]:
# ============================================================
# GRAFICO — ONE-HOT + ESTIMADOR HIBRIDO, MAE E R² POR AREA (LOGO)
# ============================================================
fig, (ax_mae, ax_r2) = plt.subplots(1, 2, figsize=(14, 5.5))

ordem_areas_oh = comparativo_onehot_areas.sort_values("MAE_logo_onehot_hibrido")["planilha"].tolist()
dados_chart_oh = comparativo_onehot_areas.set_index("planilha").loc[ordem_areas_oh]
x = np.arange(len(ordem_areas_oh))
cor_onehot_hibrido = "#1baf7a"

barras_mae = ax_mae.bar(x, dados_chart_oh["MAE_logo_onehot_hibrido"], color=cor_onehot_hibrido)
ax_mae.bar_label(barras_mae, fmt="%.2f", padding=2, fontsize=8)
ax_mae.set_xticks(x)
ax_mae.set_xticklabels(ordem_areas_oh, rotation=20, ha="right")
ax_mae.set_ylabel("MAE de teste, LOGO (°C)")
ax_mae.set_title("MAE por área (quanto menor, melhor ↓)", fontsize=11, fontweight="bold")
for lado in ("top", "right"):
    ax_mae.spines[lado].set_visible(False)

barras_r2 = ax_r2.bar(x, dados_chart_oh["R2_logo_onehot_hibrido"], color=cor_onehot_hibrido)
ax_r2.bar_label(barras_r2, fmt="%.2f", padding=2, fontsize=8)
ax_r2.set_xticks(x)
ax_r2.set_xticklabels(ordem_areas_oh, rotation=20, ha="right")
ax_r2.set_ylabel("R² de teste, LOGO")
ax_r2.set_title("R² por área (quanto maior, melhor ↑)", fontsize=11, fontweight="bold")
ax_r2.axhline(0, color="#595959", linewidth=1)
for lado in ("top", "right"):
    ax_r2.spines[lado].set_visible(False)

fig.suptitle("One-hot encoding + estimador híbrido nas sete áreas (LOGO)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### Leitura do resultado

A tabela acima aplica one-hot encoding + estimador híbrido nas sete áreas, sempre avaliado por LOGO (equivalente a Leave-One-Out nesta base de 32 linhas por área).

- **R² varia bastante entre áreas**: de 0,12 (`oven_hmi`) e 0,16 (`ipc_comp`) até 0,51 (`mainback`) e 0,67 (`coil_temps`) — nas áreas mais fracas, o modelo explica pouco da variância mesmo com one-hot + híbrido.
- **MAE mais baixo em `main_vent_sys`** (1,96 °C) e mais alto em `coil_temps` (10,54 °C) — a escala de erro aceitável difere por área, então R² e MAE devem ser lidos junto com a faixa de temperatura de cada termopar.
- **Média das sete áreas**: R² 0,362 — o ganho de one-hot + híbrido segue concentrado nas áreas onde a codificação ordinal impunha mais distorção (como `coil_temps`, puxada por `run_case`).

**Ressalva de método.** A busca de hiperparâmetros no espaço one-hot (Célula 25b) usou uma grade menor que `GRADE_HIPERPARAMETROS` (teto de 300 árvores em vez de 800), para manter viável o tempo de execução do LOGO com estimador híbrido nas sete áreas — a primeira tentativa com a grade cheia foi interrompida após mais de 1h30 sem terminar.

**Isto não muda o modelo de produção.** É uma comparação de codificação de features; qualquer adoção de one-hot + híbrido no modelo real depende de decisão de arquitetura fora deste notebook.


## 22. Explicabilidade do modelo (SHAP): como e por que o modelo chega ao resultado

As seções anteriores comparam **erro** entre variantes (média, mediana, híbrido; ordinal x one-hot), mas nenhuma delas abre a caixa-preta: até aqui não há resposta para *quais variáveis pesam mais* nem para *como o valor de uma variável específica empurra a temperatura prevista para cima ou para baixo*. Esta seção usa [SHAP](https://github.com/shap/shap) (SHapley Additive exPlanations, Lundberg & Lee, 2017) para isso, aplicado à combinação **one-hot encoding + estimador híbrido**. 

**O que o SHAP calcula.** Para cada predição, o SHAP reparte a diferença entre essa predição e uma predição-base (a média das predições no conjunto usado para treinar o modelo) entre as variáveis de entrada, de forma que a soma das partes seja exatamente igual a essa diferença. Isso serve dois propósitos:

- **Global**: qual variável, em média, mais afasta as predições da base
- **Local**: para UMA predição específica, quanto cada variável contribuiu, empurrou a temperatura prevista para cima ou para baixo, e por quanto.

**O algoritmo `TreeExplainer`.** `RandomForestRegressor` é uma floresta de árvores; para essa família de modelo o SHAP tem um algoritmo exato e rápido (`TreeExplainer`) que percorre a estrutura da árvore em vez de reamostrar features como as versões genéricas fariam.

**Ressalva importante.** O `TreeExplainer` calcula os valores de Shapley sobre a agregação padrão do `RandomForestRegressor`: a média das previsões de cada árvore, cada árvore usando a média da própria folha. O estimador híbrido da Seção 18 (`prever_hibrido`) troca essa média pela mediana em folhas cuja média cai fora do [Q1, Q3] — uma agregação diferente, sem suporte nativo no `TreeExplainer`. Por isso, nesta seção:

- a explicabilidade **global** usa o `TreeExplainer` sobre a agregação padrão — uma leitura válida de quais variáveis o modelo usa para particionar o espaço, independente de qual estatística de folha é usada depois para prever;
- a explicabilidade **local** (uma predição específica) é feita **duas vezes** — uma para `.predict()` via `TreeExplainer`, outra para `prever_hibrido` via um explicador de caixa-preta (`shap.explainers.Permutation`, mais lento mas sem depender da estrutura interna da árvore) — para conferir se a leitura muda quando a agregação muda.

**Dependência nova.** Esta seção usa a biblioteca `shap`, que não é usada em nenhuma outra parte do repositório.


In [ ]:
# ============================================================
# CÉLULA 26a — MODELO FINAL POR ÁREA (ESPAÇO ONE-HOT) E IMPORTÂNCIA SHAP GLOBAL
# Um modelo por área, treinado em TODAS as linhas disponíveis (não só as 70%
# de treino da Célula 4), com os hiperparâmetros já escolhidos na Célula 25b.
# O objetivo aqui não é medir erro — isso já foi feito em LOGO nas Células
# 25b/25c — e sim descrever o que o modelo, treinado com todo o dado
# disponível (o que se faria antes de um deploy real), aprendeu.
# ============================================================
try:
    import shap
except ImportError:
    %pip install -q shap
    import shap


def importancia_shap_por_variavel(valores_shap, colunas, variaveis):
    """|SHAP| médio por variável original.

    `valores_shap` tem forma (amostras, colunas_one_hot, alvos). Para cada
    variável original, soma (com sinal) o SHAP das colunas one-hot que
    pertencem a ela -- convenção usual para reagrupar SHAP de uma variável
    categórica dummificada -- e só then toma o valor absoluto, tirando a
    média nas amostras e nos alvos. Fazer o valor absoluto ANTES da média
    evita que a contribuição em um alvo cancele a de outro alvo de sinal
    oposto.
    """
    importancias = {}
    for variavel in variaveis:
        indices = [i for i, c in enumerate(colunas) if c.startswith(variavel + "_")]
        soma_dummies = valores_shap[:, indices, :].sum(axis=1)   # (amostras, alvos), com sinal
        importancias[variavel] = np.abs(soma_dummies).mean()
    return pd.Series(importancias).sort_values(ascending=False)


MODELOS_ONEHOT_FINAIS = {}
IMPORTANCIA_SHAP_POR_AREA = {}

for insumo in INSUMOS_LOTE:
    nome_area = insumo["nome_area"]
    X_oh, Y_oh, alvos_oh = DADOS_ONEHOT_POR_AREA[nome_area]
    params_oh = PARAMS_ONEHOT_POR_AREA[nome_area]

    modelo_final = RandomForestRegressor(random_state=SEMENTE, n_jobs=-1, **params_oh)
    modelo_final.fit(X_oh, Y_oh)
    MODELOS_ONEHOT_FINAIS[nome_area] = modelo_final

    explicador = shap.TreeExplainer(modelo_final)
    valores_shap = explicador.shap_values(X_oh)

    # Normaliza o formato entre versões do shap: lista (uma matriz por alvo)
    # ou array já empilhado.
    if isinstance(valores_shap, list):
        valores_shap = np.stack(valores_shap, axis=-1)   # -> (amostras, colunas, alvos)
    elif valores_shap.ndim == 2:
        valores_shap = valores_shap[:, :, None]

    IMPORTANCIA_SHAP_POR_AREA[nome_area] = importancia_shap_por_variavel(
        valores_shap, list(X_oh.columns), VARIAVEIS_INDEPENDENTES)

    print(f"OK   {nome_area}  (colunas one-hot: {X_oh.shape[1]}, alvos: {len(alvos_oh)})")

tabela_shap_areas = pd.DataFrame(IMPORTANCIA_SHAP_POR_AREA).T
tabela_shap_areas.index = [a.replace("_data.csv", "") for a in tabela_shap_areas.index]

print("\n" + "=" * 100)
print("IMPORTÂNCIA SHAP GLOBAL (|SHAP| médio em °C, agregado nas colunas one-hot de cada variável e nos alvos)")
print("=" * 100)
print(tabela_shap_areas.round(3).to_string())


### Leitura: participação relativa, não magnitude absoluta

A tabela acima está em °C de contribuição média (`|SHAP|`), então áreas com termopares em patamares de temperatura mais altos tendem a ter números maiores só por causa da escala — não é diretamente comparável entre áreas. O gráfico abaixo normaliza cada área para somar 100%, para comparar a **participação relativa** de cada variável dentro de cada área, independente da escala de temperatura.


In [ ]:
# ============================================================
# CÉLULA 26b — MAPA DE CALOR: PARTICIPAÇÃO DE CADA VARIÁVEL NA IMPORTÂNCIA
# SHAP, POR ÁREA (cada linha soma 100%)
# ============================================================
tabela_shap_norm = tabela_shap_areas.div(tabela_shap_areas.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(9.5, 6))
im = ax.imshow(tabela_shap_norm.values, cmap="YlOrRd", aspect="auto", vmin=0)

ax.set_xticks(range(len(tabela_shap_norm.columns)))
ax.set_xticklabels(tabela_shap_norm.columns, rotation=35, ha="right")
ax.set_yticks(range(len(tabela_shap_norm.index)))
ax.set_yticklabels(tabela_shap_norm.index)

limite_cor = tabela_shap_norm.values.max()
for i in range(tabela_shap_norm.shape[0]):
    for j in range(tabela_shap_norm.shape[1]):
        valor = tabela_shap_norm.values[i, j]
        cor_texto = "white" if valor > limite_cor * 0.6 else "#333333"
        ax.text(j, i, f"{valor:.0%}", ha="center", va="center", fontsize=8, color=cor_texto)

ax.set_title("Participação de cada variável na importância SHAP, por área\n(cada linha soma 100%)",
             fontsize=11, fontweight="bold")
fig.colorbar(im, ax=ax, label="participação na importância total da área")
plt.tight_layout()
plt.show()

print("Variável mais importante por área:")
print(tabela_shap_norm.idxmax(axis=1).to_string())


## 23. Explicabilidade local: por que a pior previsão de `coil_temps` saiu daquele jeito

A parte global respondeu "o que o modelo usa, em média, para prever". Esta seção responde "por que ESTA previsão específica saiu com este valor" — para o caso de maior erro absoluto encontrado no conjunto de teste de `coil_temps_data.csv`, a área usada como exemplo-guia. Escolher o pior caso, e não um caso típico, é deliberado: é onde entender o "porquê" do modelo importa mais na prática.


In [ ]:
# ============================================================
# CÉLULA 27a — ESCOLHA DO CASO: MAIOR ERRO ABSOLUTO, ESTIMADOR HÍBRIDO,
# CONJUNTO DE TESTE DE coil_temps_data.csv
# Usa o mesmo corte de teste da Célula 4 (`X_te` em INSUMOS_LOTE), agora no
# espaço one-hot: como `construir_onehot_completo` e `treinar_area` aplicam
# a mesma sequência de identificar_alvos -> filtrar_por_cobertura -> dropna
# (sem resetar o índice), as linhas de teste caem nos mesmos índices originais
# nos dois espaços, então dá para reutilizar `insumo["X_te"].index` para
# selecionar as mesmas linhas em `X_oh`.
# ============================================================
AREA_EXEMPLO = "coil_temps_data.csv"
CAMINHO_EXEMPLO = next(c for c in PLANILHAS if c.endswith(AREA_EXEMPLO))
insumo_exemplo = next(i for i in INSUMOS_LOTE if i["nome_area"] == AREA_EXEMPLO)
X_oh_exemplo, Y_oh_exemplo, alvos_exemplo = DADOS_ONEHOT_POR_AREA[AREA_EXEMPLO]
modelo_exemplo = MODELOS_ONEHOT_FINAIS[AREA_EXEMPLO]

indices_teste = insumo_exemplo["X_te"].index
X_te_oh = X_oh_exemplo.loc[indices_teste]
Y_te_oh = Y_oh_exemplo.loc[indices_teste]

previsao_hibrida_teste = prever_hibrido(modelo_exemplo, X_oh_exemplo, Y_oh_exemplo, X_te_oh)
erro_absoluto = np.abs(previsao_hibrida_teste - Y_te_oh.values)
erro_absoluto_valido = np.where(np.isnan(erro_absoluto), -np.inf, erro_absoluto)

linha_idx, alvo_idx = np.unravel_index(np.argmax(erro_absoluto_valido), erro_absoluto.shape)
LINHA_EXEMPLO = X_te_oh.index[linha_idx]
ALVO_EXEMPLO = alvos_exemplo[alvo_idx]

x_linha = X_te_oh.loc[[LINHA_EXEMPLO]]
valor_medido = float(Y_te_oh.loc[LINHA_EXEMPLO, ALVO_EXEMPLO])
valor_previsto_padrao = float(modelo_exemplo.predict(x_linha)[0, alvo_idx])
valor_previsto_hibrido = float(previsao_hibrida_teste[linha_idx, alvo_idx])

# Configuração em texto legível (não one-hot), para descrever o caso.
_coil_bruto = pd.read_csv(CAMINHO_EXEMPLO)
_alvos_coil_leg, _ = identificar_alvos(_coil_bruto, VARIAVEIS_INDEPENDENTES)
_alvos_coil_leg, _ = filtrar_alvos_por_cobertura(_coil_bruto, _alvos_coil_leg, COBERTURA_MINIMA_ALVO)
_coil_tratado_leg = padronizar_categorias(
    _coil_bruto.dropna(subset=_alvos_coil_leg), VARIAVEIS_INDEPENDENTES, GRAFIAS_DIVERGENTES)
configuracao_exemplo = _coil_tratado_leg.loc[LINHA_EXEMPLO, VARIAVEIS_INDEPENDENTES].to_dict()

print(f"Caso escolhido: {AREA_EXEMPLO}, linha (índice original) {LINHA_EXEMPLO}, termopar {ALVO_EXEMPLO!r}")
print("Configuração:")
for variavel, valor in configuracao_exemplo.items():
    print(f"   {variavel:24s} = {valor}")
print(f"\nMedido:              {valor_medido:7.2f} °C")
print(f"Previsto (padrão):   {valor_previsto_padrao:7.2f} °C   (erro {valor_previsto_padrao - valor_medido:+.2f} °C)")
print(f"Previsto (híbrido):  {valor_previsto_hibrido:7.2f} °C   (erro {valor_previsto_hibrido - valor_medido:+.2f} °C)")


### Decomposição SHAP da previsão padrão (`.predict()`)

O gráfico abaixo é um *waterfall*: parte da predição-base (`E[f(X)]`, a média das predições no conjunto usado para treinar o modelo) e soma a contribuição de cada variável até chegar na predição final para este caso. Barras vermelhas empurram a temperatura prevista para cima, azuis para baixo.


In [ ]:
# ============================================================
# CÉLULA 27b — WATERFALL SHAP, PREVISÃO PADRÃO (.predict())
# ============================================================
explicador_exemplo = shap.TreeExplainer(modelo_exemplo)
explicacao = explicador_exemplo(x_linha)

if explicacao.values.ndim == 3:
    explicacao_alvo = explicacao[0, :, alvo_idx]
else:
    explicacao_alvo = explicacao[0]

shap.plots.waterfall(explicacao_alvo, show=False)
plt.title(f"SHAP (predict padrão) — {ALVO_EXEMPLO}, {AREA_EXEMPLO.replace('_data.csv', '')}",
          fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()


### A mesma decomposição, agora para o estimador híbrido

Repete a explicação local, desta vez tratando `prever_hibrido` como caixa-preta (`shap.explainers.Permutation`) — mais lento (percorre a função várias vezes, em vez de usar a estrutura da árvore diretamente), mas válido para qualquer função de previsão, não só para `.predict()`.


In [ ]:
# ============================================================
# CÉLULA 27c — WATERFALL SHAP, ESTIMADOR HÍBRIDO (shap.explainers.Permutation)
# prever_hibrido não tem suporte no TreeExplainer (ver ressalva na Célula 22),
# então é tratado como caixa-preta: o explicador só enxerga entrada -> saída.
# ============================================================
def prever_hibrido_alvo(X_input, modelo, X_treino_ref, Y_treino_ref, indice_alvo):
    """Wrapper de uma única saída de `prever_hibrido`, para o SHAP tratar
    como caixa-preta: recebe uma matriz de linhas no espaço one-hot e devolve
    a previsão híbrida só do termopar `indice_alvo`."""
    X_df = pd.DataFrame(np.asarray(X_input), columns=X_treino_ref.columns)
    return prever_hibrido(modelo, X_treino_ref, Y_treino_ref, X_df)[:, indice_alvo]


funcao_hibrida = lambda X_input: prever_hibrido_alvo(
    X_input, modelo_exemplo, X_oh_exemplo, Y_oh_exemplo, alvo_idx)

fundo_permutacao = shap.sample(X_oh_exemplo, min(20, len(X_oh_exemplo)), random_state=SEMENTE)
explicador_hibrido = shap.explainers.Permutation(funcao_hibrida, fundo_permutacao, seed=SEMENTE)
explicacao_hibrida = explicador_hibrido(x_linha)

shap.plots.waterfall(explicacao_hibrida[0], show=False)
plt.title(f"SHAP (estimador híbrido) — {ALVO_EXEMPLO}, {AREA_EXEMPLO.replace('_data.csv', '')}",
          fontsize=10, fontweight="bold")
plt.tight_layout()
plt.show()

comparacao_local = pd.DataFrame({
    "SHAP_predict_padrao": np.asarray(explicacao_alvo.values),
    "SHAP_hibrido": np.asarray(explicacao_hibrida.values[0]),
}, index=x_linha.columns)
comparacao_local["diferenca"] = comparacao_local["SHAP_hibrido"] - comparacao_local["SHAP_predict_padrao"]
comparacao_local = comparacao_local.reindex(
    comparacao_local["SHAP_hibrido"].abs().sort_values(ascending=False).index)

print("Comparação da decomposição local: predict padrão x estimador híbrido")
print(comparacao_local.round(3).to_string())
print(f"\nBase (predict padrão):  {float(np.asarray(explicacao_alvo.base_values)):.2f} °C")
print(f"Base (híbrido):         {float(np.asarray(explicacao_hibrida.base_values)[0]):.2f} °C")
print(f"Previsto padrão:  {valor_previsto_padrao:.2f} °C   |   "
      f"Previsto híbrido: {valor_previsto_hibrido:.2f} °C   |   Medido: {valor_medido:.2f} °C")


### Leitura do resultado

- **Global (heatmap).** A importância SHAP confirma, por um caminho diferente: `run_case` é a variável dominante em `coil_temps`, e por larga margem — sua contribuição SHAP ali (~18,5 °C de `|SHAP|` médio) é a maior de toda a tabela, cerca de 5x a segunda maior contribuição da tabela. Nas demais áreas a variável dominante varia: `blower_direction` lidera em `mainback` e `oven_hmi`; `run_case` continua à frente em `main_vent_sys`, `ct_knobs` e `ipc_cooling`; e `deflector_hole` supera `run_case` por pouco em `ipc_comp`.
- **Local (pior erro do teste em `coil_temps`, termopar `Aluminium Plate FR`).** Tanto o predict padrão (96,85 °C) quanto o híbrido (97,50 °C) subestimam bastante o valor medido (110,80 °C) — o híbrido melhora pouco este caso específico (0,65 °C), não o "resolve". As duas decomposições concordam que `run_case` e `material_coil_support` dominam a explicação, mas **divergem no ranking**: no predict padrão, a maior contribuição isolada vem de `material_coil_support_Aluminium` (-2,74 °C); no híbrido, as categorias de `run_case` dominam com folga (`run_case_Broil C2`: +5,82 °C; `run_case_Broil Qualitative`: -5,47 °C), e a contribuição de `run_case_Pyro` cai a zero. Essa divergência é evidência concreta — não só uma ressalva teórica — de que `TreeExplainer` (sobre `.predict()`) e a explicação de caixa-preta do híbrido não são intercambiáveis.


## Conclusão: o que foi testado e o que fica recomendado

Este notebook partiu de um Random Forest simples por área e testou cinco
caminhos para melhorá-lo:

1. **Tuning de hiperparâmetros** (busca em grade): ganho pequeno e instável —
   em várias áreas o ajuste apenas reorganizou ruído, como a Seção 12 explica
   com apoio na literatura.
2. **Remoção de variáveis pouco importantes** (`board_position` e
   `material_coil_support`): simplifica o modelo, mas piora o erro na maioria
   das áreas — o MAE só cai em 2 das 7 (diferença média +0,648 °C) e o R²
   melhora em apenas 1 das 7. Não fica recomendada.
3. **Alinhar o critério do modelo à métrica cobrada** (variante B,
   `criterion="absolute_error"`): uma única linha de mudança, que melhorou o
   erro médio em 5 das 7 áreas.
4. **Estimador híbrido média/mediana por folha**: escolhe média ou mediana
   caso a caso; foi o melhor dos três critérios em 4 das 7 áreas e melhor que
   a média em 6 das 7.
5. **Validação Leave-One-Group-Out em vez de holdout fixo**: não muda o
   modelo, muda a forma de medi-lo — com 32–43 linhas por área, um único corte
   de 20% dá números pouco confiáveis, e o LOGO dá uma leitura mais estável.

**Recomendação final.** Duas propostas ficaram validadas e nenhuma delas
domina a outra nos resultados obtidos aqui, então a escolha não é decidida por
este notebook e fica pendente de validação com o time:

- **Variante B (critério MAE)** — mudança mínima, uma linha de código, usa
  apenas o que o `scikit-learn` já oferece. Mais fácil de manter e explicar.
  Melhorou em 5 das 7 áreas.
- **Estimador híbrido** — melhor resultado em mais áreas quando comparado aos
  três critérios lado a lado, mas exige código próprio (não é nativo da
  biblioteca), o que aumenta o custo de manutenção e de auditoria.

O trade-off, em uma frase: **B é mais simples de sustentar, o híbrido tende a
errar menos em mais áreas**. Combinar os dois (híbrido sobre um modelo já
treinado com critério MAE) não foi testado aqui e seria o próximo experimento
natural.

Vale manter em vista o limite que atravessa todas as seções: o volume de dados
por área é pequeno, e nenhum ajuste de modelo substitui novos ensaios físicos
ou de simulação — é o teto estrutural já registrado no risco R02.


## Fontes

Referências usadas na Seção 12 (por que o R² e o MAE se comportam assim):

[Intrinsic stability of RF variable importance](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4739337/) ·

[Hypothesis testing with random forests](https://arxiv.org/pdf/1904.07830) ·

[RF Regression in Python (GeeksforGeeks)](https://www.geeksforgeeks.org/machine-learning/random-forest-regression-in-python/) ·

[Position_Salaries.csv (10 linhas)](https://github.com/qtle3/random-forest-regressor) ·

[R² shrinkage e tamanho de amostra (Minitab)](https://blog.minitab.com/en/blog/adventures-in-statistics-2/r-squared-shrinkage-and-power-and-sample-size-guidelines-for-regression-analysis) ·

[max_features e decorrelação (scikit-learn)](https://inria.github.io/scikit-learn-mooc/python_scripts/ensemble_random_forest.html)